# Track B — Fine-tune Relation Classifier

Phân loại quan hệ **6 lớp** giữa hai atomic claim của hai reviewer:
`AGREEMENT · PARTIAL_AGREEMENT · COMPLEMENTARY · PARTIAL_CONTRADICTION · CONTRADICTION · UNRELATED`

- **Train** trên `pipeline_data/processed/trackB_silver.jsonl` — nhãn LLM, tiếng Anh, review paper ICLR.
- **Chấm điểm cuối** trên `pipeline_data/golden_set/gold_test.jsonl` — 129 cặp `HUMAN_VERIFIED`,
  tiếng Việt, phản biện đề tài đại học. Đây là con số duy nhất có nền người.

Contract nhãn: [`docs/RUBRIC_GOLD.md`](../docs/RUBRIC_GOLD.md) — rút ra từ chính tập gold.

> ⚠ **Notebook sẽ dừng ở mục 3** nếu `trackB_silver.jsonl` chưa được gán lại theo contract
> gold. Rubric cũ gán `COMPLEMENTARY` cho *"cùng aspect nhưng khác điểm cụ thể"* (494/496 cặp),
> còn gold gọi đúng tình huống đó là `UNRELATED`. Lớp này chiếm 45% dữ liệu train, nên train
> trước khi gán lại là dạy model trả lời ngược đáp án. Chạy `src/data/relabel_complementary.py`.

---

### Bốn lựa chọn thiết kế không phải mặc định

1. **Đối xứng theo cấu trúc.** Quan hệ không phụ thuộc claim nào đứng trước. Toàn bộ sự cố
   order-flip của pipeline ensemble (94% cặp phải đi debate chỉ vì có model đổi ý khi đảo A/B)
   đến từ chỗ này. Xử lý bằng **augment 2 chiều lúc train** + **cộng logit 2 chiều lúc suy luận**,
   và **đo trực tiếp flip-rate** để kiểm chứng chứ không chỉ hi vọng.
2. **Trọng số lớp.** CONTRADICTION chỉ ~2.6%. Không có trọng số thì model bỏ hẳn lớp này
   mà accuracy vẫn đẹp.
3. **Split đọc từ đĩa.** `src/train/split.py` chia theo nhóm paper + stratified theo nhãn +
   ghim few-shot vào train. Notebook **không** tự chia lại.
4. **Backbone đa ngữ.** `xlm-roberta-base` chứ không phải `roberta-base` — tập gold là tiếng
   Việt. Đây là đánh giá cross-lingual zero-shot: train EN, chấm VI.

### Thứ tự chạy
`Setup → Data → Split → Baselines → Train → Đánh giá → 5-fold CV → Learning curve → GOLD → Lưu checkpoint`

## 0 · Setup

In [1]:
# Pin 4.44.2 đã fail build wheel cho tokenizers -> notebook âm thầm chạy bản Colab có sẵn,
# tức phiên bản KHÔNG tái lập được. Không pin nữa, nhưng IN RA bản thật để ghi vào manifest.
!pip install -q -U transformers scikit-learn
import torch, transformers, sklearn, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(f"torch {torch.__version__} | transformers {transformers.__version__} "
      f"| sklearn {sklearn.__version__} | {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.0 MB/s eta 0:00:00
name, memory.total [MiB]
Tesla T4, 15360 MiB

torch 2.11.0+cu128 | transformers 5.16.1 | sklearn 1.9.0 | Tesla T4


## 1 · Lấy dữ liệu

Repo public nên clone thẳng. **Nhớ push `trackB_silver.jsonl` lên GitHub trước khi chạy cell này.**

In [2]:
import os, json
REPO = "https://github.com/navihat/peer-review-claim-relations.git"
if not os.path.exists("peer-review-claim-relations"):
    !git clone -q {REPO}
%cd peer-review-claim-relations
!git pull -q
SILVER = "pipeline_data/processed/trackB_silver.jsonl"

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

rows = read_jsonl(SILVER)
print(f"{len(rows)} cặp | {len({r['paper_id'].split('|')[0] for r in rows})} nhóm paper")

/content/peer-review-claim-relations
989 cặp | 210 nhóm paper


## 2 · Cấu hình

In [3]:
from dataclasses import dataclass, replace

# ---------------------------------------------------------------------------
# BA MỨC ĐỘ PHÂN GIẢI NHÃN — chọn bằng LABEL_VIEW
# ---------------------------------------------------------------------------
# Cùng một dữ liệu, đọc ở ba độ mịn. Không phải để chọn cái nào cho điểm đẹp — mỗi mức
# trả lời một câu hỏi khác nhau, và BÁO CÁO PHẢI IN CẢ BA (xem mục 12).
#
#   full6    contract nghiên cứu, khớp RUBRIC_GOLD.md. Con số trung thực nhất, và là con
#            số thấp nhất: macro-F1 chia đều cho lớp CONTRADICTION 22 mẫu / F1 0.000
#            (TRAIN.md muc 11.6 — lớp đó "bơm nhiễu vào macro-F1 chứ không đóng góp").
#
#   merge5   gộp PARTIAL_CONTRADICTION vào CONTRADICTION. Đây là phép gộp DUY NHẤT có
#            cơ sở cấu trúc: cả hai nằm CÙNG nhánh cùng-issue, cùng chiều, chỉ khác MỨC
#            ĐỘ — tức một trục độ thật, không phải hai lớp khác nhau. Kéo lớp đó từ 22
#            lên ~182 mẫu CV. Đo được +0.048 (muc 11.5) và +0.044 (muc 12.4) trên HAI
#            checkpoint khác nhau — phép gộp duy nhất lặp lại được.
#            ⚠ KHÔNG gộp PARTIAL_AGREEMENT vào AGREEMENT dù nó ăn +0.027: P_AGREE nằm ở
#            nhánh KHÁC-issue (prior gold 18.6+14.0 = 32.6%, khớp đúng số đo ở muc 12.5),
#            nên gộp nó là vượt Q1 — đúng ranh giới đang gánh 0.581 lỗi.
#
#   deploy3  độ mịn mà hệ thống downstream thật sự tiêu thụ: hai phản biện ĐỒNG THUẬN /
#            MÂU THUẪN / nói hai chuyện khác nhau. Có cơ sở đo được chứ không phải né:
#            learning curve phẳng từ n≈400 (muc 12.6) -> dữ liệu đã nói hết ở độ mịn 6
#            lớp; sai Q2 bất động 0.16-0.22 qua mọi lần chạy (muc 11.5, 12.5) -> ở mức
#            thô model ổn định. Đây là con số để trả lời "dùng được chưa".
#
# ⚠ Δ macro-F1 giữa các mức KHÔNG dùng để chọn bộ nhãn. Gộp hai lớp mà model đang lẫn thì
#   điểm luôn tăng bất kể phép gộp có nghĩa hay không — bằng chứng: gộp COMPLEMENTARY vào
#   UNRELATED ăn +0.062 (muc 12.4), cao hơn mọi phương án khác, mà đó là phép nhập hai
#   nhánh Q1 tại đúng chỗ chúng chia đôi. Bảng Δ đo ĐỘ LẪN CỦA MODEL, không đo bài toán.
LABEL_VIEW = "full6"          # "full6" | "merge5" | "deploy3"

FULL6 = ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY",
         "PARTIAL_CONTRADICTION","CONTRADICTION","UNRELATED"]

TO_MERGE5  = {"PARTIAL_CONTRADICTION": "CONTRADICTION"}
TO_DEPLOY3 = {"AGREEMENT":"CONSENSUS", "PARTIAL_AGREEMENT":"CONSENSUS",
              "COMPLEMENTARY":"CONSENSUS", "PARTIAL_CONTRADICTION":"CONFLICT",
              "CONTRADICTION":"CONFLICT", "UNRELATED":"UNRELATED"}

VIEW_FN = {"full6":  lambda l: l,
           "merge5": lambda l: TO_MERGE5.get(l, l),
           "deploy3": lambda l: TO_DEPLOY3.get(l, l)}
VIEW_LABELS = {"full6":  FULL6,
               "merge5": ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY",
                          "CONTRADICTION","UNRELATED"],
               "deploy3": ["CONSENSUS","CONFLICT","UNRELATED"]}
VIEW_ORDER = ["full6","merge5","deploy3"]     # mịn -> thô

def view_label(l, view=None):
    """Ánh xạ một nhãn full6 (hoặc merge5) sang mức đang chọn."""
    return VIEW_FN[view or LABEL_VIEW](l)

LABELS = VIEW_LABELS[LABEL_VIEW]
L2I = {l:i for i,l in enumerate(LABELS)}
ALL_LABELS = FULL6

# Kết quả gom lại cho bảng báo cáo ở mục 12. Mỗi cell tự ghi vào đây khi chạy xong.
REPORT = {}

# Trục quan hệ: sai giữa hai nhãn KỀ NHAU nhẹ hơn nhiều so với sai giữa hai nhãn XA NHAU.
# UNRELATED nằm ngoài trục, chỉ kề COMPLEMENTARY.
AXIS = [l for l in ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY",
                    "PARTIAL_CONTRADICTION","CONTRADICTION"] if l in LABELS]

def axis_dist(a, b):
    """Khoảng cách trên trục quan hệ; UNRELATED cách COMPLEMENTARY 1 bước."""
    if a == b: return 0
    if "UNRELATED" in (a, b):
        other = b if a == "UNRELATED" else a
        return 1 if other == "COMPLEMENTARY" else 3
    return abs(AXIS.index(a) - AXIS.index(b))

# ---------------------------------------------------------------------------
# Phân rã Q1/Q2 — thước đo mức nặng KHỚP VỚI CONTRACT, đọc thay cho axis_dist
# ---------------------------------------------------------------------------
# RUBRIC_GOLD.md muc 1: annotator quyết định theo HAI câu hỏi độc lập, không theo một trục
#     Q1  hai claim có nhắm tới CÙNG MỘT issue cụ thể không?  -> chọn NHÁNH
#     Q2  quan hệ giữa hai phán xét trong nhánh đó là gì?     -> chọn NHÃN
# Điểm bất ngờ của cây đó: PARTIAL_AGREEMENT nằm ở nhánh KHÁC-issue, cùng chỗ với
# UNRELATED — không phải cạnh AGREEMENT.
#
# Nên axis_dist chấm NGƯỢC ở đúng hai ô lỗi lớn nhất đã đo (TRAIN.md muc 11.5):
#     COMPLEMENTARY <-> UNRELATED      axis_dist 1 "nhẹ"   thực ra SAI Q1 (vượt nhánh)
#     PARTIAL_AGREEMENT <-> UNRELATED  axis_dist 3 "nặng"  thực ra chỉ SAI Q2 (cùng nhánh)
# Giữ axis_dist để còn so với các lần chạy cũ, nhưng số để ĐỌC là q1_q2_split().
SAME_ISSUE = {"AGREEMENT","COMPLEMENTARY","PARTIAL_CONTRADICTION","CONTRADICTION"}

# Ở deploy3 thì CONSENSUS trải trên CẢ HAI nhánh (AGREEMENT/COMPLEMENTARY là cùng-issue,
# PARTIAL_AGREEMENT là khác-issue), nên phân rã Q1/Q2 mất định nghĩa. merge5 thì vẫn chuẩn:
# PARTIAL_CONTRADICTION và CONTRADICTION đều ở nhánh cùng-issue.
Q1_DEFINED = LABEL_VIEW in ("full6", "merge5")

def branch_of(label):
    return "CUNG_ISSUE" if label in SAME_ISSUE else "KHAC_ISSUE"

def q1_q2_split(y_true, y_pred, tag=""):
    """Tách lỗi: sai Q1 = nhầm nhánh cùng/khác issue; sai Q2 = đúng nhánh, sai nhãn."""
    if not Q1_DEFINED:
        print(f"{tag}(bỏ qua: ở LABEL_VIEW={LABEL_VIEW} nhánh Q1 không còn định nghĩa được)")
        return None
    ok = q1 = q2 = 0
    for t, p in zip(y_true, y_pred):
        a, b = LABELS[t], LABELS[p]
        if a == b:                          ok += 1
        elif branch_of(a) != branch_of(b):  q1 += 1
        else:                               q2 += 1
    n = len(y_true)
    print(f"{tag}đúng {ok/n:.3f}  |  sai Q1 (nhầm nhánh) {q1/n:.3f} ({q1}/{n})"
          f"  |  sai Q2 (đúng nhánh) {q2/n:.3f}")
    return ok/n, q1/n, q2/n


# ---------------------------------------------------------------------------
# eval_views — CHẤM CÙNG MỘT DỰ ĐOÁN Ở MỌI MỨC THÔ HƠN, trong một lần chạy
# ---------------------------------------------------------------------------
# Không phải train lại ba lần: model xuất nhãn ở LABEL_VIEW, còn hàm này đọc lại chính
# dự đoán đó ở độ mịn thô hơn. Đúng như hệ thống thật sẽ làm — checkpoint trả 6 lớp, bên
# gọi chỉ cần biết "đồng thuận hay mâu thuẫn".
def eval_views(y_true, y_pred, tag="", store=None):
    """y_true/y_pred là CHỈ SỐ trong LABELS hiện tại. In macro-F1 + acc ở mọi mức thô hơn."""
    i0 = VIEW_ORDER.index(LABEL_VIEW)
    t_names = [LABELS[i] for i in y_true]
    p_names = [LABELS[i] for i in y_pred]
    # Cột cuối = majority của CHÍNH tập đang chấm (oracle: dùng nhãn thật của nó để biết
    # lớp nào đông nhất). Khác với DÒNG "majority" ở mục 4 và mục 11, vốn đoán lớp đông
    # nhất của TRAIN — đó mới là luật triển khai được. Hai số này lệch nhau là bình thường.
    print(f"\n{tag}{'mức nhãn':<12}{'n lớp':>7}{'macro-F1':>11}{'acc':>9}{'maj(oracle)':>13}")
    out = {}
    for v in VIEW_ORDER[i0:]:
        labs = VIEW_LABELS[v]; idx = {l:i for i,l in enumerate(labs)}
        yt = np.array([idx[view_label(l, v)] for l in t_names])
        yp = np.array([idx[view_label(l, v)] for l in p_names])
        # labels=TẤT CẢ lớp của view, không để sklearn tự lấy hợp của y_true/y_pred.
        # Nếu không: dòng "majority" chỉ đoán 1 lớp nên macro của nó chia cho ít lớp hơn
        # dòng model (model đoán nhiều lớp, kể cả lớp vắng mặt trong y_true) -> hai dòng
        # trong cùng một bảng lại chia cho mẫu số khác nhau và hết so được với nhau.
        _all = list(range(len(labs)))
        maj = np.full(len(yt), collections.Counter(yt).most_common(1)[0][0])
        m  = f1_score(yt, yp,  labels=_all, average="macro", zero_division=0)
        a  = (yt == yp).mean()
        mj = f1_score(yt, maj, labels=_all, average="macro", zero_division=0)
        print(f"{' '*len(tag)}{v:<12}{len(labs):>7}{m:>11.4f}{a:>9.4f}{mj:>13.4f}")
        out[v] = {"macro_f1": float(m), "acc": float(a), "majority_f1": float(mj)}
    if store is not None:
        REPORT[store] = out
    return out


@dataclass
class Cfg:
    # mDeBERTa-v3-base-xnli: backbone chính cho workflow. Đa ngữ + NLI warm-up.
    # Cross-lingual: train EN silver, chấm VI gold. Thắng xlm-r-base ở cả test-EN
    # lẫn gold (so sánh ở muc 9.b). Xem docs/TRAIN.md muc 15.
    model: str = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

    # 6 -> 20 epoch, patience 2 -> 4. Lần chạy trước KHÔNG HỘI TỤ và đó là nguyên nhân
    # lớn nhất kéo mọi con số xuống (TRAIN.md muc 11.1):
    #   - cùng 6 epoch: roberta-base về loss 0.3395, xlm-r mới tới 0.7267
    #   - trong CV các fold dừng ở loss 0.90-1.37, mà đoán bừa = ln(6) = 1.79
    #     -> mới đi được 29-61% quãng đường tới hội tụ
    #   - fold ÍT train nhất (loss 1.37) lại cho macro-F1 CAO nhất (0.354); tương quan
    #     loss<->F1 là +0.45 -> chênh lệch giữa các fold là nhiễu, không phải học
    # xlm-r cần nhiều epoch hơn roberta ở cùng lr. Đặt trần cao rồi để val chọn epoch.
    # MỐC KIỂM TRA: loss cuối phải xuống vùng 0.3-0.5. Chưa tới thì tăng lr lên 3e-5 và
    # chạy lại, ĐỪNG đọc kết quả — train_model sẽ tự in cảnh báo nếu còn cao.
    epochs: int = 20
    patience: int = 4
    lr: float = 2e-5

    batch: int = 16
    eval_batch: int = 64
    max_len: int = 160
    fp16: bool = True
    symmetric_aug: bool = True     # train: nhân đôi cặp theo 2 thứ tự
    symmetric_tta: bool = True     # infer: TRUNG BÌNH logit 2 thứ tự (xem predict_logits)
    seed: int = 20260828

    # --- Chọn epoch (sửa sau lần chạy 2026-08-29c, xem TRAIN.md muc 12.2) ---
    # Lần đó val chọn epoch 10 (val macro-F1 0.3741, cao nhất) nhưng epoch đó có train loss
    # 0.0557 — đã thuộc lòng. Hệ quả: val TĂNG còn test 0.2932 -> 0.2265 và gold
    # 0.2548 -> 0.1971, ECE 0.309 -> 0.523. Val chỉ 100 cặp và có CONT:2, nên chỉ cần model
    # bắt trúng 1 trong 2 mẫu CONTRADICTION là macro-F1 val nhảy tới ±0.11 — lớn hơn cả
    # khoảng cách giữa các epoch. Hai chốt chặn (bản trước có thêm loss_floor — hằng số đó
    # được chọn bằng cách nhìn gold, nên đã bỏ hẳn, xem TRAIN.md muc 14.2):
    #   min_val_support : lớp có ít hơn ngần này mẫu trong val bị LOẠI khỏi tiêu chí chọn
    #                     epoch (vẫn được train và vẫn được chấm ở mọi mục khác)
    #   min_delta       : cải thiện nhỏ hơn ngần này coi như không cải thiện -> ưu tiên
    #                     epoch SỚM khi hai epoch ngang nhau, tức ít thuộc lòng hơn
    # Dừng sớm khi MỘT trong hai tín hiệu chững lại sau `patience` epoch liền: val_sel
    # (theo min_delta ở trên) HOẶC val loss không giảm nữa (xem train_model, val_loss_bad).
    min_val_support: int = 5
    min_delta: float = 0.01

    # Bỏ 110 cặp sinh bằng luật "ghép claim hai paper khác nhau -> UNRELATED". Chúng đã bị
    # loại khỏi silver lúc gán lại; cờ này giữ để chặn nếu chúng quay lại.
    drop_synthetic_unrelated: bool = True

    # Bù lệch tiên nghiệm lúc suy luận (dùng ở mục 10). Đo được trên gold: silver có
    # 34.6% UNRELATED, gold chỉ 14.0%, mà model đoán UNRELATED 35.7% -> nó chép gần
    # nguyên prior của tập train. 0.0 = tắt, 1.0 = bù đủ. Xem prior_shift_logits().
    prior_tau: float = 1.0

    # Ghi lại mức nhãn đang dùng để nó đi kèm checkpoint (labels.json ở mục 11).
    label_view: str = LABEL_VIEW

    # --- Tỉ lệ nhãn: chỉnh bằng TRỌNG SỐ, không bằng cách vứt dữ liệu ---
    # Silver có 34.6% UNRELATED, gold chỉ 14.0%, và muc 11.4 đo được model đoán UNRELATED
    # 35.7% trên gold — nó chép gần nguyên prior của tập train. w ∝ 1/p_train ("uniform")
    # bù QUÁ TAY theo hướng ngược lại (gold không hề đều: UNRELATED 14%, COMPLEMENTARY
    # 22.5%), nhưng vẫn AN TOÀN hơn hai lựa chọn khác: prior_shift_logits ở mục 10 (chỉnh
    # lúc SUY LUẬN, đảo ngược được — nhưng hỏng khi logit bão hoà, xem muc 12.3) hoặc
    # resample tập train (KHÔNG khuyến nghị: vứt nhãn thật để làm đẹp một phân bố).
    #
    # Từng có lựa chọn thứ ba "gold_dev" (w ∝ p_gold_dev/p_train, đọc từ 2/3 cohort gold,
    # không đụng hold-out). ĐÃ BỎ ở TRAIN.md muc 16: cohort DEV/HOLD-OUT gộp lại thành một
    # tập báo cáo duy nhất (129 cặp), nên không còn tập dev riêng để lấy prior mà không
    # đụng vào chính số báo cáo.
    weight_target: str = "uniform"

cfg = Cfg()
cfg_xlmr = replace(cfg, model="FacebookAI/xlm-roberta-base")
print(cfg)
print(f"\nLABEL_VIEW = {LABEL_VIEW} -> {len(LABELS)} lớp: {LABELS}")
if LABEL_VIEW != "full6":
    print(f"  ánh xạ: " + ", ".join(f"{l}->{view_label(l)}" for l in FULL6 if view_label(l) != l))


Cfg(model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7', epochs=20, patience=4, lr=2e-05, batch=16, eval_batch=64, max_len=160, fp16=True, symmetric_aug=True, symmetric_tta=True, seed=20260828, min_val_support=5, min_delta=0.01, drop_synthetic_unrelated=True, prior_tau=1.0, label_view='full6', weight_target='uniform')

LABEL_VIEW = full6 -> 6 lớp: ['AGREEMENT', 'PARTIAL_AGREEMENT', 'COMPLEMENTARY', 'PARTIAL_CONTRADICTION', 'CONTRADICTION', 'UNRELATED']


## 3 · Đọc split đã chia sẵn

**Notebook không tự chia dữ liệu.** Split do [`src/train/split.py`](../src/train/split.py) sinh ra
và commit vào repo, nên notebook, script train và mọi lần chạy sau đều dùng đúng một bộ.

Bản trước của notebook có bản sao riêng của `assign_groups` và tự chia tại chỗ — đó là bản
**chưa stratified**, khiến test chỉ có 3 mẫu AGREEMENT / 2 mẫu CONTRADICTION và 5-fold lệch
13 lần ở CONTRADICTION. Mọi số đo ra từ bản đó không so sánh được với nhau.

Split hiện tại đảm bảo ba điều, kiểm bằng `assert` ở dưới:

1. **Nhóm theo paper** — không paper nào nằm ở hai phần (cặp chéo paper lấy paper trái làm khoá).
2. **Stratified theo nhãn** — lệch lớn nhất so với phân bố toàn cục là 2.2 điểm %.
3. **Few-shot ghim vào train** — 39 cặp `fewshot_human_pairs` chính là các ví dụ đã dùng để
   gán nhãn 1060 cặp còn lại, nên không được phép nằm ở phần chấm điểm.

Chi tiết: [`pipeline_data/reports/split_report.md`](../pipeline_data/reports/split_report.md).

In [4]:
import random, collections
import numpy as np

SPLITS = "pipeline_data/processed/splits"

def group_of(r): return r["paper_id"].split("|")[0]

def keep(part):
    return [r for r in part
            if not (cfg.drop_synthetic_unrelated and r["source"] == "synthetic_cross_paper")]

n_before   = len(rows)
rows       = keep(rows)
train_rows = keep(read_jsonl(f"{SPLITS}/trackB_train.jsonl"))
val_rows   = keep(read_jsonl(f"{SPLITS}/trackB_val.jsonl"))
test_rows  = keep(read_jsonl(f"{SPLITS}/trackB_test.jsonl"))
FOLDS      = json.load(open(f"{SPLITS}/folds.json", encoding="utf-8"))

# --- CHỐT CHẶN: silver phải được gán lại theo contract gold trước khi train ---
# Rubric cũ gán COMPLEMENTARY cho "cùng aspect nhưng KHÁC điểm cụ thể" (494/496 cặp),
# gold gọi đúng tình huống đó là UNRELATED. COMPLEMENTARY chiếm 45% dữ liệu train, nên
# train trước khi gán lại = dạy model trả lời ngược đáp án gold. Xem docs/RUBRIC_GOLD.md.
prov = collections.Counter(r.get("provenance") for r in rows)
n_unrel = sum(1 for r in rows if r["relation"] == "UNRELATED")
if "relabel_to_gold_contract_v1" not in prov or n_unrel < 30:
    raise SystemExit(
        f"\n  DỪNG: trackB_silver.jsonl chưa được gán lại theo contract gold.\n"
        f"  provenance hiện tại: {dict(prov)}\n"
        f"  UNRELATED còn {n_unrel} cặp (cần >=30 sau khi gán lại).\n\n"
        f"  Chạy trước:\n"
        f"    python src/data/relabel_complementary.py         # xuất 10 batch x 50 cặp\n"
        f"    <gán nhãn, gộp thành decisions.jsonl>\n"
        f"    python src/data/relabel_complementary.py --apply decisions.jsonl\n"
        f"    python src/train/split.py                        # chia lại sau khi silver đổi\n")

# --- Áp LABEL_VIEW ngay tại chỗ đọc: mọi thứ phía sau chỉ đọc r["relation"] và LABELS,
#     nên gộp ở đây là gộp cho cả train, class-weight, CV, gold và checkpoint. ---
if LABEL_VIEW != "full6":
    _n6 = collections.Counter(r["relation"] for r in rows)
    for _part in (rows, train_rows, val_rows, test_rows):
        for r in _part: r["relation"] = view_label(r["relation"])
    print(f"[LABEL_VIEW={LABEL_VIEW}] gộp {len(FULL6)} -> {len(LABELS)} lớp: "
          + ", ".join(f"{l}={sum(v for k,v in _n6.items() if view_label(k)==l)}" for l in LABELS))

def subsample_groups(part, frac, seed=cfg.seed):
    """Lấy ~frac dữ liệu nhưng CẮT THEO NHÓM PAPER, không cắt giữa nhóm (cho learning curve)."""
    by = collections.defaultdict(list)
    for r in part: by[group_of(r)].append(r)
    g = sorted(by); random.Random(seed).shuffle(g)
    out, target = [], frac*len(part)
    for k in g:
        if len(out) >= target: break
        out.extend(by[k])
    return out

def summarize(name, part):
    d = collections.Counter(r["relation"] for r in part)
    print(f"{name:<7}{len(part):>5}  {len({group_of(r) for r in part}):>4} paper  " +
          "  ".join(f"{l[:4]}:{d.get(l,0)}" for l in LABELS))

for n_, p_ in [("train",train_rows),("val",val_rows),("test",test_rows)]: summarize(n_, p_)

gt,gv,gs = ({group_of(r) for r in p} for p in (train_rows,val_rows,test_rows))
assert not (gt&gv) and not (gt&gs) and not (gv&gs), "paper lọt sang phần khác"
assert not [r for r in val_rows + test_rows if r["source"] == "fewshot_human_pairs"], \
    "cặp few-shot lọt vào phần chấm điểm"
assert len(train_rows)+len(val_rows)+len(test_rows) == len(rows), \
    "split không phủ kín silver — chạy lại src/train/split.py sau khi silver đổi"
print("\n[OK] không paper nào nằm ở hai phần")
print("[OK] few-shot chỉ nằm trong train")
print("[OK] silver đã theo contract gold")

# Bản trước in "đã bỏ synthetic_cross_paper: 989 -> 989 cặp" ngay cả khi không bỏ cặp nào,
# nghe như bộ lọc đang hoạt động trong khi thực ra 110 cặp đó đã bị loại từ lúc gán lại.
n_dropped = n_before - len(rows)
if n_dropped:
    print(f"[!] đã bỏ {n_dropped} cặp synthetic_cross_paper: {n_before} -> {len(rows)}")
else:
    print(f"[OK] silver không còn synthetic_cross_paper (đã loại lúc gán lại) — {len(rows)} cặp")

# Lệch tiên nghiệm giữa train và gold: ghi ra ngay từ đây vì nó chi phối cách đọc mục 10.
_pt = collections.Counter(r["relation"] for r in train_rows)
_PG6 = {"UNRELATED":.140, "COMPLEMENTARY":.225, "PARTIAL_AGREEMENT":.186,
        "PARTIAL_CONTRADICTION":.186, "AGREEMENT":.155, "CONTRADICTION":.109}
_pg = collections.Counter()
for _l, _v in _PG6.items(): _pg[view_label(_l)] += _v
print(f"\n{'lớp':<24}{'prior train':>12}{'prior gold':>12}{'chênh':>9}")
for l in LABELS:
    a = _pt.get(l,0)/len(train_rows)
    print(f"{l:<24}{a:>11.1%}{_pg[l]:>12.1%}{a-_pg[l]:>+9.1%}")
print("-> mục 10 bù chênh này bằng prior_shift_logits(); xem docs/TRAIN.md muc 10.6 & 11.3")
print("[!] test chỉ ~110 cặp -> đọc số từ 5-fold CV; số CUỐI CÙNG lấy từ gold ở mục 10")


train    788   180 paper  AGRE:61  PART:173  COMP:122  PART:134  CONT:25  UNRE:273
val      100    16 paper  AGRE:7  PART:22  COMP:16  PART:17  CONT:2  UNRE:36
test     101    14 paper  AGRE:8  PART:22  COMP:15  PART:17  CONT:2  UNRE:37

[OK] không paper nào nằm ở hai phần
[OK] few-shot chỉ nằm trong train
[OK] silver đã theo contract gold
[OK] silver không còn synthetic_cross_paper (đã loại lúc gán lại) — 989 cặp

lớp                      prior train  prior gold    chênh
AGREEMENT                      7.7%       15.5%    -7.8%
PARTIAL_AGREEMENT             22.0%       18.6%    +3.4%
COMPLEMENTARY                 15.5%       22.5%    -7.0%
PARTIAL_CONTRADICTION         17.0%       18.6%    -1.6%
CONTRADICTION                  3.2%       10.9%    -7.7%
UNRELATED                     34.6%       14.0%   +20.6%
-> mục 10 bù chênh này bằng prior_shift_logits(); xem docs/TRAIN.md muc 10.6 & 11.3
[!] test chỉ ~110 cặp -> đọc số từ 5-fold CV; số CUỐI CÙNG lấy từ gold ở mục 10


## 3.b · Gold: nạp và kiểm độ dài

**Gold là tập báo cáo duy nhất — không dùng để chọn tham số hay ngưỡng.**
Toàn bộ 129 cặp được dùng để đánh giá cuối cùng.

In [5]:
from transformers import AutoTokenizer

GOLD = "pipeline_data/golden_set/gold_test.jsonl"
gold_raw = read_jsonl(GOLD)

gold_rows = [{"pair_id": r["example_id"], "paper_id": r["cohort_id"],
              "aspect": r["criterion_id"], "source": "gold", "cohort": r["cohort_id"],
              "left":  {"text": r["left"]["text"],  "stance": r["left"]["stance"]},
              "right": {"text": r["right"]["text"], "stance": r["right"]["stance"]},
              "relation": view_label(r["expected_relation"])} for r in gold_raw]
assert all(r["relation"] in L2I for r in gold_rows), \
    "gold co nhan ngoai LABELS -- kiem tra LABEL_VIEW / docs/RUBRIC_GOLD.md"

_n_cohort = len({r["cohort"] for r in gold_rows})
print(f"gold {len(gold_rows)} cap / {_n_cohort} cohort")
d = collections.Counter(r["relation"] for r in gold_rows)
print("  " + "  ".join(f"{l[:4]}:{d.get(l,0)}" for l in LABELS))

# --- Kiem do dai: gold dai hon silver, cap gold co bi cat cut khong? ---
_tok_chk = AutoTokenizer.from_pretrained(cfg.model)
_hdr = ("tap".ljust(10) + "n".rjust(5) + "median".rjust(9)
        + "p90".rjust(7) + "max".rjust(7)
        + ("bi cat @" + str(cfg.max_len)).rjust(14))
print("\n" + _hdr)
_trunc = {}
for nm, part in [("train", train_rows), ("test", test_rows), ("gold", gold_rows)]:
    L = sorted([len(_tok_chk(r["left"]["text"], r["right"]["text"])["input_ids"]) for r in part])
    cut = sum(1 for x in L if x > cfg.max_len) / len(L)
    _trunc[nm] = cut
    print(f"{nm:<10}{len(L):>5}{L[len(L)//2]:>9}{L[int(len(L)*.9)]:>7}{L[-1]:>7}{cut:>13.1%}")
del _tok_chk

_gap = _trunc["gold"] - _trunc["train"]
if _gap > 0.05:
    print("\n  Warning gold bi cat nhieu hon train %.1f%% -- TANG cfg.max_len roi chay lai." % (_gap*100,))
else:
    print("\n  [OK] ti le bi cat cua gold khong vuot train qua 5 diem -- max_len du.")


gold 129 cap / 3 cohort
  AGRE:20  PART:24  COMP:29  PART:24  CONT:14  UNRE:18


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]


tap           n   median    p90    max   bi cat @160
train       788       54     83    162         0.1%
test        101       59     91    124         0.0%
gold        129      132    184    211        27.9%

  Warning gold bi cat nhieu hon train 27.8% -- TANG cfg.max_len roi chay lai.


## 4 · Baseline — mốc để so sánh

Không có mốc thì macro-F1 = 0.45 là tốt hay tệ đều không biết.

- **majority**: luôn đoán COMPLEMENTARY (lớp đông nhất)
- **stance-rule**: dùng đúng tín hiệu đã dùng để đào cặp — stance đối nghịch → PARTIAL_CONTRADICTION

In [6]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

y_test = np.array([L2I[r["relation"]] for r in test_rows])

# Bản trước ghim cứng majority = COMPLEMENTARY. Đó là tàn dư từ TRƯỚC đợt gán lại ở muc
# 10, khi COMPLEMENTARY còn chiếm 45%; sau khi gán lại thì lớp đông nhất là UNRELATED
# (34.6%). Ghim sai lớp làm baseline YẾU đi một cách giả tạo — tức thổi phồng khoảng cách
# giữa model và baseline. Lấy đúng lớp đông nhất của tập train.
MAJ_LABEL = collections.Counter(r["relation"] for r in train_rows).most_common(1)[0][0]
print(f"majority = lớp đông nhất của train = {MAJ_LABEL}\n")

def stance_rule(r):
    s = {r["left"]["stance"], r["right"]["stance"]}
    if s == {"POSITIVE","NEGATIVE"}: return L2I[view_label("PARTIAL_CONTRADICTION")]
    if s == {"POSITIVE"}:            return L2I[view_label("PARTIAL_AGREEMENT")]
    return L2I[view_label("COMPLEMENTARY")]

def run_baselines(part, tag):
    """Chấm hai baseline trên MỘT tập bất kỳ, ở mọi mức nhãn."""
    yt = np.array([L2I[r["relation"]] for r in part])
    out = {}
    for name, p in [("majority",    np.full(len(part), L2I[MAJ_LABEL])),
                    ("stance-rule", np.array([stance_rule(r) for r in part]))]:
        print(f"\n--- {name} trên {tag} (n={len(part)}) ---")
        out[name] = eval_views(yt, p, store=f"baseline:{name}:{tag}")
    return out

run_baselines(test_rows,  "test-EN")
run_baselines(gold_rows,  "gold")

majority = lớp đông nhất của train = UNRELATED


--- majority trên test-EN (n=101) ---

mức nhãn      n lớp   macro-F1      acc  maj(oracle)
full6             6     0.0894   0.3663       0.0894
merge5            5     0.1072   0.3663       0.1072
deploy3           3     0.1787   0.3663       0.2055

--- stance-rule trên test-EN (n=101) ---

mức nhãn      n lớp   macro-F1      acc  maj(oracle)
full6             6     0.1597   0.2376       0.0894
merge5            5     0.1983   0.2475       0.1072
deploy3           3     0.3872   0.5149       0.2055

--- majority trên gold (n=129) ---

mức nhãn      n lớp   macro-F1      acc  maj(oracle)
full6             6     0.0408   0.1395       0.0612
merge5            5     0.0490   0.1395       0.0910
deploy3           3     0.0816   0.1395       0.2409

--- stance-rule trên gold (n=129) ---

mức nhãn      n lớp   macro-F1      acc  maj(oracle)
full6             6     0.1079   0.2558       0.0612
merge5            5     0.2079   0.3333       0.09

{'majority': {'full6': {'macro_f1': 0.04081632653061224,
   'acc': 0.13953488372093023,
   'majority_f1': 0.06118143459915612},
  'merge5': {'macro_f1': 0.04897959183673469,
   'acc': 0.13953488372093023,
   'majority_f1': 0.09101796407185628},
  'deploy3': {'macro_f1': 0.08163265306122448,
   'acc': 0.13953488372093023,
   'majority_f1': 0.24092409240924093}},
 'stance-rule': {'full6': {'macro_f1': 0.10786151731033622,
   'acc': 0.2558139534883721,
   'majority_f1': 0.06118143459915612},
  'merge5': {'macro_f1': 0.20786519332142306,
   'acc': 0.3333333333333333,
   'majority_f1': 0.09101796407185628},
  'deploy3': {'macro_f1': 0.38468720821661995,
   'acc': 0.6356589147286822,
   'majority_f1': 0.24092409240924093}}}

## 4.b · Baseline NLI zero-shot — *fine-tune có hơn model có sẵn không?*

Phép thử gắt nhất trong [`TRAIN.md` muc 5](../docs/TRAIN.md), treo qua 5 mục chưa chạy.
Nếu một model NLI tải về dùng luôn cũng ngang ngửa thì toàn bộ công gán nhãn là vô ích —
cần biết điều đó **trước** khi báo cáo.

In [7]:
# mDeBERTa-v3-base-xnli: chính model đã dùng ở B2-guard. Đa ngữ nên đọc được cả gold
# tiếng Việt — đây là "base model gốc" đúng nghĩa để so.
#
# Ánh xạ NLI -> quan hệ: chỉ tự nhiên ở mức deploy3, và đó là điểm đáng nói chứ không
# phải điểm yếu của phép so. NLI có đúng 3 nhãn:
#     entailment    -> CONSENSUS        hai claim củng cố nhau
#     contradiction -> CONFLICT         hai claim chỏi nhau
#     neutral       -> UNRELATED        không suy ra được gì từ nhau
# Không ngưỡng, không tham số — nên không có chỗ nào để tinh chỉnh cho vừa kết quả.
# Ở full6/merge5 thì model có sẵn KHÔNG diễn đạt nổi contract (nó không có khái niệm
# "cùng chiều nhưng khác phạm vi"), và bản thân việc đó đã là một lý lẽ cho fine-tune.
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

from transformers import AutoModelForSequenceClassification as _AMSC
# device được đặt ở mục 5 (sau cell này) — tự lo lấy để mục 4.b chạy độc lập được.
device = globals().get("device") or torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = _AMSC.from_pretrained(NLI_MODEL).to(device).eval()
_id2 = {i: l.lower() for i, l in nli.config.id2label.items()}
_col = {v: k for k, v in _id2.items()}          # đọc thứ tự nhãn từ config, không đoán
print("nhãn NLI:", nli.config.id2label)

@torch.no_grad()
def nli_probs(rows_, bs=32):
    """Trung bình xác suất NLI của CẢ HAI thứ tự — quan hệ đối xứng, giống cfg.symmetric_tta."""
    acc = None
    for lo, ro in [(0,1), (1,0)]:
        parts = []
        for i in range(0, len(rows_), bs):
            ch = rows_[i:i+bs]
            t = [(r["left"]["text"], r["right"]["text"]) for r in ch]
            enc = nli_tok([x[lo] for x in t], [x[ro] for x in t], truncation=True,
                          max_length=cfg.max_len, padding=True, return_tensors="pt").to(device)
            parts.append(torch.softmax(nli(**enc).logits.float(), dim=1).cpu())
        p = torch.cat(parts)
        acc = p if acc is None else acc + p
    return (acc / 2).numpy()

_D3 = VIEW_LABELS["deploy3"]; _D3I = {l:i for i,l in enumerate(_D3)}
_NLI2REL = {"entailment": "CONSENSUS", "contradiction": "CONFLICT", "neutral": "UNRELATED"}

def eval_nli(part, tag):
    p = nli_probs(part)
    pred = np.array([_D3I[_NLI2REL[_id2[int(i)]]] for i in p.argmax(1)])
    true = np.array([_D3I[view_label(r["relation"], "deploy3")] for r in part])
    maj  = np.full(len(true), collections.Counter(true).most_common(1)[0][0])
    _all3 = list(range(3))
    m  = f1_score(true, pred, labels=_all3, average="macro", zero_division=0)
    a  = (true == pred).mean()
    mj = f1_score(true, maj, labels=_all3, average="macro", zero_division=0)
    print(f"\n--- NLI zero-shot trên {tag} (n={len(part)}, mức deploy3) ---")
    print(classification_report(true, pred, labels=list(range(3)),
                                target_names=_D3, digits=3, zero_division=0))
    print(f"macro-F1={m:.4f}  acc={a:.4f}  (majority cùng tập = {mj:.4f})")
    print("phân bố NLI đoán: " + ", ".join(
        f"{_D3[i]}={int((pred==i).sum())}" for i in range(3)))
    REPORT[f"nli:{tag}"] = {"deploy3": {"macro_f1": float(m), "acc": float(a),
                                        "majority_f1": float(mj)}}
    return m

for _tag, _part in [("test-EN", test_rows), ("gold", gold_rows)]:
    eval_nli(_part, _tag)

del nli; torch.cuda.empty_cache()
print("\n-> so với dòng deploy3 của model fine-tune ở mục 7, 8 và 10. Đây là con số trả lời")
print("   'fine-tune có hơn model có sẵn không'. Bảng gom đủ ở mục 12.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

nhãn NLI: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

--- NLI zero-shot trên test-EN (n=101, mức deploy3) ---
              precision    recall  f1-score   support

   CONSENSUS      0.000     0.000     0.000        45
    CONFLICT      0.714     0.263     0.385        19
   UNRELATED      0.370     0.919     0.527        37

    accuracy                          0.386       101
   macro avg      0.361     0.394     0.304       101
weighted avg      0.270     0.386     0.265       101

macro-F1=0.3039  acc=0.3861  (majority cùng tập = 0.2055)
phân bố NLI đoán: CONSENSUS=2, CONFLICT=7, UNRELATED=92

--- NLI zero-shot trên gold (n=129, mức deploy3) ---
              precision    recall  f1-score   support

   CONSENSUS      0.833     0.137     0.235        73
    CONFLICT      0.632     0.316     0.421        38
   UNRELATED      0.173     0.944     0.293        18

    accuracy                          0.302       129
   macro avg      0.546     0.466     0.316       129
weight

In [8]:
from transformers import AutoModelForSequenceClassification as _AMSC

_NLI2REL = {"entailment": "CONSENSUS", "contradiction": "CONFLICT", "neutral": "UNRELATED"}
_NLI2M5  = {"entailment": "PARTIAL_AGREEMENT", "contradiction": "CONTRADICTION", "neutral": "UNRELATED"}

if "nli" not in globals():
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli     = _AMSC.from_pretrained(NLI_MODEL).to(device).eval()
    _id2    = {i: l.lower() for i, l in nli.config.id2label.items()}

_D3  = VIEW_LABELS["deploy3"];  _D3I = {l: i for i, l in enumerate(_D3)}
_M5  = VIEW_LABELS["merge5"];   _M5I = {l: i for i, l in enumerate(_M5)}

def _eval_nli_full(part, tag):
    p       = nli_probs(part)
    raw_cls = [_id2[int(i)] for i in p.argmax(1)]

    d3p  = np.array([_D3I[_NLI2REL[c]] for c in raw_cls])
    d3t  = np.array([_D3I[view_label(r["relation"], "deploy3")] for r in part])
    d3m  = np.full(len(d3t), collections.Counter(d3t).most_common(1)[0][0])
    d3a  = list(range(len(_D3)))
    d3f1 = f1_score(d3t, d3p, labels=d3a, average="macro", zero_division=0)
    d3ac = float((d3t == d3p).mean())
    d3mf = float(f1_score(d3t, d3m, labels=d3a, average="macro", zero_division=0))

    m5p  = np.array([_M5I[_NLI2M5[c]] for c in raw_cls])
    m5t  = np.array([_M5I[view_label(r["relation"], "merge5")] for r in part])
    m5m  = np.full(len(m5t), collections.Counter(m5t).most_common(1)[0][0])
    m5a  = list(range(len(_M5)))
    m5f1 = f1_score(m5t, m5p, labels=m5a, average="macro", zero_division=0)
    m5ac = float((m5t == m5p).mean())
    m5mf = float(f1_score(m5t, m5m, labels=m5a, average="macro", zero_division=0))

    REPORT[f"nli:{tag}"] = {
        "deploy3": {"macro_f1": float(d3f1), "acc": d3ac, "majority_f1": d3mf},
        "merge5":  {"macro_f1": float(m5f1), "acc": m5ac, "majority_f1": m5mf},
    }
    print(f"\n--- NLI zero-shot trên {tag} ---")
    print(f"  deploy3  macro-F1={d3f1:.4f}")
    print(f"  merge5   macro-F1={m5f1:.4f}  (mapping cứng — xấp xỉ)")

for _tag, _part in [("test-EN", test_rows), ("gold", gold_rows)]:
    _eval_nli_full(_part, _tag)

@torch.no_grad()
def _nli_raw_pred(rows_):
    out = []
    for i in range(0, len(rows_), cfg.eval_batch):
        ch  = rows_[i:i+cfg.eval_batch]
        t   = [(r["left"]["text"], r["right"]["text"]) for r in ch]
        enc = nli_tok([x[0] for x in t], [x[1] for x in t],
                      truncation=True, max_length=cfg.max_len,
                      padding=True, return_tensors="pt").to(device)
        out.append(nli(**enc).logits.float().cpu().argmax(1))
    return torch.cat(out).numpy()

_fwd_nli  = _nli_raw_pred(test_rows)
_swp_nli  = [{**r, "left": r["right"], "right": r["left"]} for r in test_rows]
_rev_nli  = _nli_raw_pred(_swp_nli)
_flip_nli = (_fwd_nli != _rev_nli)
REPORT["flip:test-EN-nli"] = float(_flip_nli.mean())
print(f"\nflip-rate NLI zero-shot (thô): {_flip_nli.mean():.3f}  ({_flip_nli.sum()}/{len(_flip_nli)})")

del nli
torch.cuda.empty_cache()
print("\n-> chạy lại cell mục 11 để cập nhật bảng tổng hợp.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


--- NLI zero-shot trên test-EN ---
  deploy3  macro-F1=0.3039
  merge5   macro-F1=0.1823  (mapping cứng — xấp xỉ)

--- NLI zero-shot trên gold ---
  deploy3  macro-F1=0.3165
  merge5   macro-F1=0.1651  (mapping cứng — xấp xỉ)

flip-rate NLI zero-shot (thô): 0.208  (21/101)

-> chạy lại cell mục 11 để cập nhật bảng tổng hợp.


## 4.c · Baseline GPT-4o-mini zero-shot

So sánh bổ sung với LLM mạnh — cho biết **trần thực tế của task** và liệu fine-tune có
vượt được zero-shot LLM không. Chạy cả 2 thứ tự (A,B) và (B,A) để đo flip-rate đồng thời
(LLM không có logit nên không average được, dùng (A,B) khi hai thứ tự bất đồng).

In [9]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [10]:
# !pip install -q openai   # bỏ comment lần đầu
import os
from openai import OpenAI

GPT_MODEL = "gpt-4o-mini"
_client   = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

_REL_DESC = """
- AGREEMENT: Both claims propose or conclude the same thing at the same scope.
- PARTIAL_AGREEMENT: Both claims point in the same direction but differ in scope or emphasis.
- COMPLEMENTARY: The claims address the same issue and complement each other without contradiction.
- PARTIAL_CONTRADICTION: The claims partially contradict — one undermines part of the other.
- CONTRADICTION: The claims directly contradict — if one is correct, the other must be wrong.
- UNRELATED: The claims address entirely different issues and cannot be directly compared."""

_SYS = (
    "You are an expert at classifying the relation between two peer-review claims about AI systems.\n"
    "Given CLAIM A and CLAIM B, classify their relation as EXACTLY one of:" + _REL_DESC
    + "\nRespond with ONLY the label name (e.g. AGREEMENT), nothing else."
)

def _gpt_call(a, b):
    r = _client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "system", "content": _SYS},
                  {"role": "user",   "content": f"CLAIM A: {a}\n\nCLAIM B: {b}"}],
        temperature=0, max_tokens=20
    )
    raw = r.choices[0].message.content.strip().upper()
    return raw if raw in set(FULL6) else next((l for l in FULL6 if l in raw), "COMPLEMENTARY")

def gpt_predict_sym(rows_, label):
    preds, flips = [], []
    for idx, r in enumerate(rows_):
        if idx % 20 == 0:
            print(f"  {label} {idx}/{len(rows_)} ...", end="\r")
        a, b = r["left"]["text"], r["right"]["text"]
        p_ab, p_ba = _gpt_call(a, b), _gpt_call(b, a)
        flips.append(p_ab != p_ba)
        preds.append(p_ab)
    print(f"  {label} {len(rows_)}/{len(rows_)} done")
    return np.array([L2I.get(p, L2I["COMPLEMENTARY"]) for p in preds]), float(np.mean(flips))

# --- Test set ---
y_pred_gpt_test, flip_test_gpt = gpt_predict_sym(test_rows, "[test-EN]")
eval_views(y_test, y_pred_gpt_test, tag="[test-EN] ", store="gpt4om:test-EN")
REPORT["flip:test-EN-gpt"] = flip_test_gpt
print(f"flip-rate GPT-4o-mini (test, thô): {flip_test_gpt:.3f}")

# --- Gold ---
y_gold_gpt_all  = np.array([L2I[r["relation"]] for r in gold_rows])
y_pred_gpt_gold, flip_gold_gpt = gpt_predict_sym(gold_rows, "[gold]  ")
eval_views(y_gold_gpt_all, y_pred_gpt_gold, tag="[gold] ", store="gpt4om:gold")
REPORT["flip:gold-gpt"] = flip_gold_gpt
print(f"flip-rate GPT-4o-mini (gold, thô): {flip_gold_gpt:.3f}")


  [test-EN] 101/101 done

[test-EN] mức nhãn      n lớp   macro-F1      acc  maj(oracle)
          full6             6     0.3267   0.5446       0.0894
          merge5            5     0.4055   0.5644       0.1072
          deploy3           3     0.6505   0.6634       0.2055
flip-rate GPT-4o-mini (test, thô): 0.267
  [gold]   129/129 done

[gold] mức nhãn      n lớp   macro-F1      acc  maj(oracle)
       full6             6     0.4131   0.4651       0.0612
       merge5            5     0.5341   0.5426       0.0910
       deploy3           3     0.6839   0.7054       0.2409
flip-rate GPT-4o-mini (gold, thô): 0.341


## 5 · Model + vòng huấn luyện

In [11]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

device = torch.device("cuda")

class PairSet(Dataset):
    def __init__(self, rows, tok, max_len, symmetric_aug):
        self.ex = []
        for r in rows:
            y = L2I[r["relation"]]; a, b = r["left"]["text"], r["right"]["text"]
            self.ex.append((a,b,y))
            if symmetric_aug: self.ex.append((b,a,y))   # cùng nhãn, đảo thứ tự
        self.tok, self.max_len = tok, max_len
    def __len__(self): return len(self.ex)
    def __getitem__(self, i):
        a,b,y = self.ex[i]
        e = self.tok(a, b, truncation=True, max_length=self.max_len, padding=False)
        e["labels"] = y; return e

def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b.pop("labels") for b in batch])
        out = tok.pad(batch, return_tensors="pt"); out["labels"] = labels
        return out
    return collate

@torch.no_grad()
def predict_logits(model, tok, rows, symmetric_tta):
    """symmetric_tta=True: TRUNG BÌNH logit của cả hai thứ tự (A,B) và (B,A).

    Trung bình chứ không phải TỔNG như bản trước. argmax không đổi nên mọi macro-F1 giữ
    nguyên — nhưng cộng hai logit làm THANG logit gấp đôi, khiến softmax ở mục 7.7 nhọn
    giả tạo: confidence bị đẩy lên và ECE đo ra tệ hơn thực tế (0.309 ở lần chạy trước).
    Trung bình giữ đúng thang, nên bảng hiệu chuẩn mới đọc được.
    """
    model.eval()
    orders = [(0,1)] + ([(1,0)] if symmetric_tta else [])
    total = None
    for lo, ro in orders:
        parts = []
        for i in range(0, len(rows), cfg.eval_batch):
            ch = rows[i:i+cfg.eval_batch]
            t = [(r["left"]["text"], r["right"]["text"]) for r in ch]
            enc = tok([x[lo] for x in t], [x[ro] for x in t], truncation=True,
                      max_length=cfg.max_len, padding=True, return_tensors="pt").to(device)
            parts.append(model(**enc).logits.float().cpu())
        lg = torch.cat(parts)
        total = lg if total is None else total + lg
    return (total / len(orders)).numpy()


def prior_shift_logits(logits, train_rows, target_prior=None, tau=1.0):
    """Bù lệch tiên nghiệm:  logit_adj = logit - tau*log p_train + tau*log p_đích

    Vì sao cần (đo được ở mục 10, lần chạy 2026-08-29): silver có 34.6% UNRELATED, gold
    chỉ 14.0%, và model đoán UNRELATED 35.7% trên gold — tức nó chép gần nguyên prior của
    tập train chứ không phải đọc ra quan hệ. Cột UNRELATED phình 2.56x. TRAIN.md muc 10.6
    đã dự báo đúng chuyện này trước khi chạy.

    target_prior=None -> prior ĐỀU. Đây là bản KHÔNG dùng nhãn: chỉ gỡ thiên lệch của tập
                         train, không lấy thông tin gì từ tập đang chấm. Số này báo cáo được.
    target_prior=dict -> phân bố đã biết của tập đích. Trên gold thì đây là số ORACLE (lấy
                         từ nhãn của chính tập đang chấm) — chỉ dùng để biết TRẦN của việc
                         bù prior, KHÔNG phải số để báo cáo.
    """
    cnt = collections.Counter(r["relation"] for r in train_rows)
    p_src = np.array([max(cnt.get(l, 0), 1) for l in LABELS], dtype=float); p_src /= p_src.sum()
    if target_prior is None:
        p_dst = np.ones(len(LABELS)) / len(LABELS)
    else:
        p_dst = np.array([max(target_prior.get(l, 0), 1e-6) for l in LABELS], dtype=float)
        p_dst /= p_dst.sum()
    return logits - tau*np.log(p_src) + tau*np.log(p_dst)


def train_model(train_rows, cfg, tag="", val_rows=None, sched_epochs=None):
    """Có val_rows -> chọn epoch theo macro-F1 BỀN VỮNG trên val VÀ theo val loss, dừng
    sớm sau cfg.patience ở tín hiệu nào tới trước (xem comment ở Cfg).

    Không có val_rows -> train đúng cfg.epochs (nhánh này dùng cho 5-fold CV; cfg.epochs =
    BEST_EPOCH đóng vai TRẦN duy nhất). Không còn sàn loss (bản trước dùng cfg.loss_floor,
    chọn giá trị bằng cách nhìn gold — đã bỏ, xem TRAIN.md muc 14): các fold hội tụ nhanh
    chậm khác nhau (lần 2026-08-29c cùng 10 epoch mà fold 2 kết ở loss 0.0448 còn fold 3 ở
    0.2490) và giờ SỐNG chung với chênh lệch đó thay vì che bằng một ngưỡng chọn tay —
    phương sai giữa các fold vì thế sẽ đo được to hơn, đúng bản chất dữ liệu nhỏ.

    Tiêu chí chọn epoch (`val_sel`) KHÔNG phải macro-F1 đủ 6 lớp. Lớp nào có dưới
    cfg.min_val_support mẫu trong val thì bị loại khỏi tiêu chí — với val hiện tại là
    CONTRADICTION (2 mẫu), nơi một mẫu lật làm macro-F1 nhảy ±0.11 và nuốt trọn mọi khác
    biệt thật giữa các epoch. Lớp bị loại vẫn được train và vẫn được chấm ở mọi mục khác;
    nó chỉ không được quyền chọn epoch.

    sched_epochs: HORIZON của lịch learning-rate, tách khỏi số epoch chạy thật. Cần vì CV
    train tới BEST_EPOCH, nhưng BEST_EPOCH đó được val chọn ở mục 6 DƯỚI một lịch dài
    cfg.epochs. Lịch ngắn hơn ⇒ lr đã decay về 0 ở cuối ⇒ hai con số hết so được.
    """
    # Seed TRONG hàm chứ không phải một lần lúc định nghĩa cell, để chênh lệch giữa các
    # fold chỉ còn do dữ liệu chứ không lẫn phương sai khởi tạo.
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); random.seed(cfg.seed)

    tok = AutoTokenizer.from_pretrained(cfg.model)
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model, num_labels=len(LABELS), ignore_mismatched_sizes=True, torch_dtype=torch.float32).to(device)
    dl = DataLoader(PairSet(train_rows, tok, cfg.max_len, cfg.symmetric_aug),
                    batch_size=cfg.batch, shuffle=True, collate_fn=make_collate(tok))
    # Trọng số lớp = p_đích / p_train, p_đích = phân bố ĐỀU (inverse-frequency của train).
    # Từng có lựa chọn "gold_dev" nhắm vào phân bố thật của gold; bỏ ở TRAIN.md muc 16 vì
    # không còn tập dev riêng để lấy prior mà không đụng vào số báo cáo. Xem cfg.weight_target.
    cnt = collections.Counter(r["relation"] for r in train_rows)
    p_tr = np.array([max(cnt.get(l, 0), 1) for l in LABELS], float); p_tr /= p_tr.sum()
    p_dst = np.ones(len(LABELS)); p_dst /= p_dst.sum()
    w_np = p_dst / p_tr; w_np *= len(LABELS) / w_np.sum()      # chuẩn hoá về trung bình 1
    w = torch.tensor(w_np, dtype=torch.float, device=device)
    if tag == "":
        print("  trọng số lớp (đích=%s): " % cfg.weight_target
              + "  ".join(f"{l[:6]}={x:.2f}" for l, x in zip(LABELS, w_np)))
    loss_fn = nn.CrossEntropyLoss(weight=w)
    steps = len(dl) * (sched_epochs or cfg.epochs)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(steps*0.1), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)

    y_val = sel_idx = None
    if val_rows:
        y_val = np.array([L2I[r["relation"]] for r in val_rows])
        vcnt  = collections.Counter(r["relation"] for r in val_rows)
        sel_idx = [L2I[l] for l in LABELS if vcnt.get(l, 0) >= cfg.min_val_support]
        if len(sel_idx) < 2: sel_idx = list(range(len(LABELS)))
        cut = [l for l in LABELS if L2I[l] not in sel_idx]
        print(f"  {tag}chọn epoch theo macro-F1 VÀ val loss trên {len(sel_idx)}/{len(LABELS)} lớp"
              + (f" — bỏ {', '.join(f'{l}(n={vcnt.get(l,0)})' for l in cut)}" if cut else "")
              + f" | biên {cfg.min_delta} | patience {cfg.patience}")

    best_sel, best_ep, best_state, last_loss, ep = -1.0, cfg.epochs, None, float("nan"), 0
    best_loss = float("nan")
    best_val_loss, val_loss_bad = float("inf"), 0
    for ep in range(1, cfg.epochs+1):
        model.train(); run = 0.0
        for batch in dl:
            batch = {k:v.to(device) for k,v in batch.items()}
            labels = batch.pop("labels")
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.fp16):
                loss = loss_fn(model(**batch).logits.float(), labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        last_loss = run/len(dl)
        msg = f"  {tag}epoch {ep}/{cfg.epochs}  loss={last_loss:.4f}"
        stop_sel = stop_vloss = False
        if val_rows:
            logits_val = predict_logits(model, tok, val_rows, cfg.symmetric_tta)
            vp     = logits_val.argmax(1)
            v_all  = f1_score(y_val, vp, average="macro", zero_division=0)
            v_sel  = f1_score(y_val, vp, labels=sel_idx, average="macro", zero_division=0)
            v_loss = loss_fn(torch.tensor(logits_val, dtype=torch.float, device=device),
                              torch.tensor(y_val, dtype=torch.long, device=device)).item()
            msg   += f"  val6={v_all:.4f}  val_sel={v_sel:.4f}  val_loss={v_loss:.4f}"
            if v_sel > best_sel + cfg.min_delta:      # biên: hoà thì giữ epoch SỚM hơn
                best_sel, best_ep, best_loss = v_sel, ep, last_loss
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
                msg += "  *"
            if v_loss < best_val_loss - 1e-4:
                best_val_loss, val_loss_bad = v_loss, 0
            else:
                val_loss_bad += 1
            stop_sel   = ep - best_ep >= cfg.patience
            stop_vloss = val_loss_bad >= cfg.patience
        print(msg)
        if stop_sel or stop_vloss:
            why = "val_sel không vượt biên" if stop_sel else "val loss không giảm"
            print(f"  {tag}dừng sớm: {why} sau {cfg.patience} epoch liền")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  {tag}-> giữ epoch {best_ep} (val_sel {best_sel:.4f}, train loss tại đó {best_loss:.4f})")
    else:
        best_ep = ep

    # CHỐT CHẶN HỘI TỤ. Lần 2026-08-29b dừng ở loss 0.90-1.37 (đoán bừa = 1.79) và mọi con
    # số sau đó nằm trong vùng nhiễu. Vùng lành mạnh bây giờ là dưới 0.6.
    if last_loss > 0.6:
        print(f"  {tag}⚠ CHƯA HỘI TỤ: loss cuối {last_loss:.4f} > 0.6 "
              f"(đoán bừa = {float(np.log(len(LABELS))):.3f}). Tăng cfg.epochs hoặc "
              f"cfg.lr lên 3e-5 rồi chạy lại — ĐỪNG đọc kết quả bên dưới.")
    return model, tok, best_ep

print("sẵn sàng")


sẵn sàng


## 6 · Train (train → chọn epoch bằng val → chấm trên test)

Bản đầu gọi `train_model(train_rows + val_rows, cfg)` — **gộp val thẳng vào train**, val
không làm nhiệm vụ của val. Cộng với 6 epoch cố định: train loss 0.24, ECE 0.338,
confidence vô dụng.

Rồi trần epoch nâng 6 → 20 vì lần 2026-08-29b **thiếu train** (loss dừng ở 0.90–1.37 khi
đoán bừa là ln 6 = 1.79). Sửa đó đúng: loss xuống được 0.03 và val lên 0.3109 → 0.3741.

> ⚠ **Nhưng lần 2026-08-29c lại hỏng ở đầu kia.** Val chọn epoch 10 (val cao nhất) mà epoch
> đó có train loss 0.0557 — đã thuộc lòng. **Val tăng còn test 0.2932 → 0.2265 và gold
> 0.2548 → 0.1971, ECE 0.309 → 0.523.** Nguyên nhân: val chỉ 100 cặp và có `CONT:2`, nên chỉ
> cần bắt trúng 1 trong 2 mẫu CONTRADICTION là macro-F1 val nhảy ±0.11 — lớn hơn mọi khác
> biệt thật giữa các epoch. Xem [`TRAIN.md`](../docs/TRAIN.md) muc 12.2.

Hai chốt chặn hiện có, đọc kèm dòng `loss=`:

| | |
|---|---|
| `min_val_support=5` | lớp dưới 5 mẫu val bị loại khỏi **tiêu chí chọn epoch** (`val_sel`). Vẫn được train, vẫn được chấm ở mọi mục khác — chỉ không được quyền chọn epoch. Notebook in rõ lớp nào bị loại |
| `min_delta=0.01` | cải thiện nhỏ hơn biên coi như không cải thiện ⇒ hai epoch ngang nhau thì **giữ epoch sớm**, tức ít thuộc lòng hơn |

Dừng sớm khi **một trong hai** tín hiệu chững lại liền `patience` epoch: `val_sel` không vượt
biên `min_delta`, HOẶC `val_loss` không giảm nữa. `loss_floor` (hằng số chọn bằng cách nhìn
gold ở lần 2026-08-29c) đã **bỏ hẳn** — xem [`TRAIN.md`](../docs/TRAIN.md) muc 14.2.

Dòng `val6` là macro-F1 đủ 6 lớp (để so với các lần chạy cũ); dòng `val_sel` là thứ thật sự
chọn epoch. Epoch được chọn ở đây (`BEST_EPOCH`) làm **trần** cho 5-fold CV ở mục 8.


In [12]:
model, tok, BEST_EPOCH = train_model(train_rows, cfg_xlmr, val_rows=val_rows)
logits_test = predict_logits(model, tok, test_rows, cfg_xlmr.symmetric_tta)
y_pred = logits_test.argmax(1)
print(f"\nmacro-F1 (test) = {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"BEST_EPOCH = {BEST_EPOCH}  -> 5-fold CV ở mục 8 sẽ train đúng {BEST_EPOCH} epoch")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  trọng số lớp (đích=uniform): AGREEM=1.21  PARTIA=0.43  COMPLE=0.60  PARTIA=0.55  CONTRA=2.94  UNRELA=0.27
  chọn epoch theo macro-F1 VÀ val loss trên 5/6 lớp — bỏ CONTRADICTION(n=2) | biên 0.01 | patience 4
  epoch 1/20  loss=1.8119  val6=0.0460  val_sel=0.0552  val_loss=1.7878  *
  epoch 2/20  loss=1.7938  val6=0.0609  val_sel=0.0730  val_loss=1.7731  *
  epoch 3/20  loss=1.7512  val6=0.2105  val_sel=0.2526  val_loss=1.7674  *
  epoch 4/20  loss=1.5758  val6=0.2461  val_sel=0.2954  val_loss=1.5316  *
  epoch 5/20  loss=1.3155  val6=0.2974  val_sel=0.3569  val_loss=1.4961  *
  epoch 6/20  loss=1.0123  val6=0.2658  val_sel=0.3190  val_loss=1.9959
  epoch 7/20  loss=0.6841  val6=0.3188  val_sel=0.3826  val_loss=2.1096  *
  epoch 8/20  loss=0.4211  val6=0.3288  val_sel=0.3946  val_loss=2.5228  *
  epoch 9/20  loss=0.2391  val6=0.3451  val_sel=0.4142  val_loss=2.5921  *
  dừng sớm: val loss không giảm sau 4 epoch liền
  -> giữ epoch 9 (val_sel 0.4142, train loss tại đó 0.2391)

macro-F1 

## 7 · Đánh giá

Bảy phép đo, mỗi phép trả lời một câu hỏi khác nhau. Đừng chỉ đọc accuracy.

### 7.1 · Per-class P/R/F1 — *lớp nào model bỏ rơi?*

Với phân bố lệch 45% / 2.6%, **accuracy vô dụng**: đoán COMPLEMENTARY hết vẫn được ~45%.
Macro-F1 là số chính, vì nó cho mọi lớp trọng số bằng nhau.

In [13]:
present = sorted(set(y_test) | set(y_pred))
print(classification_report(y_test, y_pred, labels=present,
      target_names=[LABELS[i] for i in present], digits=3, zero_division=0))
print(f"macro-F1={f1_score(y_test,y_pred,average='macro',zero_division=0):.4f}   "
      f"micro-F1={f1_score(y_test,y_pred,average='micro',zero_division=0):.4f}")
eval_views(y_test, y_pred, store="model:test-EN")


                       precision    recall  f1-score   support

            AGREEMENT      0.500     0.250     0.333         8
    PARTIAL_AGREEMENT      0.400     0.545     0.462        22
        COMPLEMENTARY      0.364     0.267     0.308        15
PARTIAL_CONTRADICTION      0.435     0.588     0.500        17
        CONTRADICTION      0.000     0.000     0.000         2
            UNRELATED      0.576     0.514     0.543        37

             accuracy                          0.465       101
            macro avg      0.379     0.361     0.358       101
         weighted avg      0.465     0.465     0.456       101

macro-F1=0.3576   micro-F1=0.4653

mức nhãn      n lớp   macro-F1      acc  maj(oracle)
full6             6     0.3576   0.4653       0.0894
merge5            5     0.4338   0.4752       0.1072
deploy3           3     0.5778   0.5941       0.2055


{'full6': {'macro_f1': 0.3575702075702076,
  'acc': 0.46534653465346537,
  'majority_f1': 0.08937198067632851},
 'merge5': {'macro_f1': 0.4338461538461538,
  'acc': 0.4752475247524752,
  'majority_f1': 0.10724637681159421},
 'deploy3': {'macro_f1': 0.5777777777777778,
  'acc': 0.594059405940594,
  'majority_f1': 0.20547945205479454}}

### 7.2 · Ma trận nhầm lẫn — *sai theo MẪU nào?*

Tìm **mẫu lỗi hệ thống**, không phải tỉ lệ %. Một ranh giới lệch đều một hướng
(ví dụ PARTIAL_CONTRADICTION luôn bị đoán thành CONTRADICTION) là dấu hiệu rubric
mờ ở đúng ranh giới đó, không phải model kém.

In [14]:
K = len(LABELS)
cm = confusion_matrix(y_test, y_pred, labels=list(range(K)))
print(" "*26 + "".join(f"{l[:6]:>8}" for l in LABELS) + "     n")
for i,l in enumerate(LABELS):
    print(f"{l:<26}" + "".join(f"{v:>8}" for v in cm[i]) + f"  {cm[i].sum():>5}")

print("\nCác ô nhầm nhiều nhất:")
err = [(cm[i][j], LABELS[i], LABELS[j]) for i in range(K) for j in range(K) if i!=j and cm[i][j]>0]
for n,t,p in sorted(err, reverse=True)[:8]:
    print(f"  {n:>3}x  {t}  ->  {p}   (cách {axis_dist(t,p)} bước trên trục)")

                            AGREEM  PARTIA  COMPLE  PARTIA  CONTRA  UNRELA     n
AGREEMENT                        2       6       0       0       0       0      8
PARTIAL_AGREEMENT                1      12       2       2       0       5     22
COMPLEMENTARY                    0       3       4       3       0       5     15
PARTIAL_CONTRADICTION            0       1       2      10       0       4     17
CONTRADICTION                    0       1       0       1       0       0      2
UNRELATED                        1       7       3       7       0      19     37

Các ô nhầm nhiều nhất:
    7x  UNRELATED  ->  PARTIAL_CONTRADICTION   (cách 3 bước trên trục)
    7x  UNRELATED  ->  PARTIAL_AGREEMENT   (cách 3 bước trên trục)
    6x  AGREEMENT  ->  PARTIAL_AGREEMENT   (cách 1 bước trên trục)
    5x  PARTIAL_AGREEMENT  ->  UNRELATED   (cách 3 bước trên trục)
    5x  COMPLEMENTARY  ->  UNRELATED   (cách 1 bước trên trục)
    4x  PARTIAL_CONTRADICTION  ->  UNRELATED   (cách 3 bước trên trụ

### 7.3 · Độ chính xác có dung sai theo trục — *sai NẶNG hay sai NHẸ?*

6 nhãn nằm trên một trục liên tục. Nhầm `PARTIAL_CONTRADICTION` ↔ `CONTRADICTION`
(kề nhau) nhẹ hơn hẳn nhầm `AGREEMENT` ↔ `CONTRADICTION` (cách 4 bước).
Accuracy thường coi hai lỗi này như nhau — đây là chỗ nó che mất sự thật.

In [15]:
d = np.array([axis_dist(LABELS[t], LABELS[p]) for t,p in zip(y_test, y_pred)])
print(f"đúng tuyệt đối (d=0)      : {(d==0).mean():.3f}")
print(f"đúng hoặc lệch 1 bước     : {(d<=1).mean():.3f}")
print(f"lệch >=2 bước (sai nặng)  : {(d>=2).mean():.3f}")
print(f"khoảng cách trung bình    : {d.mean():.3f} bước")
print("\nphân bố khoảng cách lỗi:", dict(collections.Counter(d.tolist())))

# Ba dòng trên giả định 6 nhãn nằm trên MỘT trục. Cây quyết định thật của contract gold
# là hai chiều, và nó chấm ngược axis_dist ở đúng hai ô lỗi lớn nhất — xem chú thích của
# q1_q2_split ở mục 2. Dòng dưới mới là thước đo khớp contract. ĐỌC DÒNG DƯỚI.
print()
q1_q2_split(y_test, y_pred, tag="theo cây RUBRIC_GOLD:  ")


đúng tuyệt đối (d=0)      : 0.465
đúng hoặc lệch 1 bước     : 0.723
lệch >=2 bước (sai nặng)  : 0.277
khoảng cách trung bình    : 1.059 bước

phân bố khoảng cách lỗi: {1: 26, 0: 47, 3: 25, 2: 3}

theo cây RUBRIC_GOLD:  đúng 0.465  |  sai Q1 (nhầm nhánh) 0.356 (36/101)  |  sai Q2 (đúng nhánh) 0.178


(0.46534653465346537, 0.3564356435643564, 0.1782178217821782)

### 7.4 · Quadratic Weighted Kappa — *hơn đoán mò bao nhiêu, có tính thứ tự?*

QWK phạt lỗi theo **bình phương khoảng cách** và hiệu chỉnh theo mức đồng thuận ngẫu nhiên.
Đây là thước đo phù hợp nhất cho nhãn có thứ tự. Bỏ UNRELATED vì nó nằm ngoài trục.

Đọc: <0.2 kém · 0.4-0.6 khá · >0.6 tốt · >0.8 rất tốt

In [16]:
from sklearn.metrics import cohen_kappa_score
ax_idx = {l:i for i,l in enumerate(AXIS)}
mask = np.array([LABELS[t] in ax_idx and LABELS[p] in ax_idx for t,p in zip(y_test,y_pred)])
if mask.sum() > 1:
    yt = [ax_idx[LABELS[t]] for t,m in zip(y_test,mask) if m]
    yp = [ax_idx[LABELS[p]] for p,m in zip(y_pred,mask) if m]
    print(f"QWK trên trục (n={mask.sum()}): {cohen_kappa_score(yt,yp,weights='quadratic'):.4f}")
print(f"Kappa thường (cả 6 lớp)  : {cohen_kappa_score(y_test,y_pred):.4f}")

QWK trên trục (n=50): 0.6539
Kappa thường (cả 6 lớp)  : 0.2946


### 7.5 · Flip-rate — *model có ĐỐI XỨNG thật không?*

**Đây là phép đo quan trọng nhất của dự án này.** Pipeline ensemble sụp đổ vì
Qwen lật nhãn 50% và Gemma 44% khi đảo A/B, khiến 94% cặp phải đi debate.

Đo trên logit THÔ (tắt TTA) để biết model tự nó đã đối xứng chưa. TTA làm flip-rate = 0
theo định nghĩa, nên nếu chỉ đo có TTA thì không phát hiện được vấn đề.

In [17]:
raw_fwd = predict_logits(model, tok, test_rows, symmetric_tta=False)
swapped = [{**r, "left": r["right"], "right": r["left"]} for r in test_rows]
raw_rev = predict_logits(model, tok, swapped, symmetric_tta=False)
f_, r_ = raw_fwd.argmax(1), raw_rev.argmax(1)
flip = (f_ != r_)
print(f"flip-rate (thô, không TTA) : {flip.mean():.3f}   ({flip.sum()}/{len(flip)} cặp)")
if flip.sum():
    dd = [axis_dist(LABELS[a], LABELS[b]) for a,b in zip(f_[flip], r_[flip])]
    print(f"  trong đó lệch 1 bước     : {sum(1 for x in dd if x==1)}/{len(dd)}")
print("\nĐối chiếu pipeline ensemble cũ: Qwen 50%, Gemma 44%, SeaLLM 18%.")
print("Dưới ~10% là đã khắc phục được vấn đề đã làm hỏng Track B.")
REPORT["flip:test-EN"] = float(flip.mean())


flip-rate (thô, không TTA) : 0.129   (13/101 cặp)
  trong đó lệch 1 bước     : 4/13

Đối chiếu pipeline ensemble cũ: Qwen 50%, Gemma 44%, SeaLLM 18%.
Dưới ~10% là đã khắc phục được vấn đề đã làm hỏng Track B.


### 7.6 · Tách theo NGUỒN dữ liệu — *con số nào là thật?*

Dữ liệu đến từ nhiều nguồn có độ tin cậy khác nhau. Bảng này bắt loại tự lừa kiểu
"model giỏi ở nguồn dễ, kém ở nguồn thật, nhưng con số tổng vẫn đẹp".

Nó đã bắt được đúng một ca như vậy: `synthetic_cross_paper` cho **acc 0.100 / macro-F1
0.045** — tệ nhất trong mọi nguồn, dù trên lý thuyết là nguồn *dễ đoán nhất*. Nguyên nhân:
lớp đó được định nghĩa bằng metadata `paper_id` mà model không bao giờ nhìn thấy
([`TRAIN.md`](../docs/TRAIN.md) muc 9.3).

Phát hiện đó ban đầu dẫn tới quyết định **bỏ hẳn lớp UNRELATED**. Quyết định ấy sau đó
**đã bị lật lại** ở muc 10.3: gold không hề coi "khác paper" là UNRELATED — cả 18 cặp
UNRELATED của gold đều cùng cohort, cùng criterion, chỉ khác reviewer. Nên lớp UNRELATED
**được giữ**, chỉ có 110 cặp sinh bằng luật là bị bỏ (`cfg.drop_synthetic_unrelated`), và
dữ liệu UNRELATED thật đến từ việc gán lại 496 cặp COMPLEMENTARY.

Vì vậy `synthetic_cross_paper` không còn xuất hiện trong bảng dưới, nhưng UNRELATED thì có.


In [18]:
by_src = collections.defaultdict(list)
for i,r in enumerate(test_rows): by_src[r["source"]].append(i)
print(f"{'nguồn':<28}{'n':>5}{'acc':>8}{'macroF1':>9}")
for s, idx in sorted(by_src.items()):
    yt_, yp_ = y_test[idx], y_pred[idx]
    print(f"{s:<28}{len(idx):>5}{(yt_==yp_).mean():>8.3f}"
          f"{f1_score(yt_,yp_,average='macro',zero_division=0):>9.3f}")

nguồn                           n     acc  macroF1
full_batch00_manual            26   0.500    0.384
full_batch01_manual            23   0.304    0.324
full_batch02_manual            22   0.455    0.276
full_batch03_manual            20   0.550    0.508
mined_stance_opposition        10   0.600    0.325


### 7.7 · Hiệu chuẩn — *có thể đặt ngưỡng ABSTAIN không?*

Nếu model tự tin sai nhiều thì không dùng được confidence để lọc. Bảng này cho biết
nên cắt ngưỡng ở đâu nếu muốn đánh đổi coverage lấy độ chính xác.

In [19]:
probs = torch.softmax(torch.tensor(logits_test), dim=1).numpy()
conf = probs.max(1); correct = (y_pred == y_test)
print(f"confidence trung bình: đúng={conf[correct].mean():.3f}  sai={conf[~correct].mean():.3f}")
print(f"\n{'ngưỡng':>8}{'coverage':>11}{'acc trên phần giữ lại':>24}")
for t in [0.0,0.5,0.6,0.7,0.8,0.9]:
    m = conf >= t
    if m.sum():
        print(f"{t:>8.1f}{m.mean():>11.3f}{correct[m].mean():>24.3f}")

# ECE 10 bin
bins = np.linspace(0,1,11); ece = 0.0
for lo,hi in zip(bins[:-1],bins[1:]):
    m = (conf>lo)&(conf<=hi)
    if m.sum(): ece += m.mean()*abs(correct[m].mean()-conf[m].mean())
print(f"\nECE = {ece:.4f}   (<0.05 hiệu chuẩn tốt, >0.15 quá tự tin)")

confidence trung bình: đúng=0.808  sai=0.748

  ngưỡng   coverage   acc trên phần giữ lại
     0.0      1.000                   0.465
     0.5      0.921                   0.505
     0.6      0.782                   0.494
     0.7      0.663                   0.537
     0.8      0.554                   0.518
     0.9      0.366                   0.432

ECE = 0.3175   (<0.05 hiệu chuẩn tốt, >0.15 quá tự tin)


## 8 · 5-fold CV — con số đáng tin trên silver

Test chỉ ~110 cặp, CONTRADICTION đúng 2 mẫu → F1 lớp đó trên test là số ngẫu nhiên.
CV cho **mỗi cặp được dự đoán đúng một lần** bởi một model chưa từng thấy nó, nên mọi lớp
đều đủ mẫu (CONTRADICTION: 22 thay vì 2).

Fold đọc từ `folds.json`, **không chia lại tại chỗ**. Bộ fold này đã stratified: CONTRADICTION
4–5 mỗi fold, trước khi sửa là 1–13. Cặp `fold = -1` là few-shot bị ghim, luôn nằm trong train
của mọi vòng và không bao giờ bị chấm điểm — nên CV chấm 950 cặp chứ không phải 989.

Mỗi vòng dựng **model mới hoàn toàn** từ checkpoint gốc, và `train_model` seed lại ngay đầu
mỗi lần gọi. Bản đầu gọi `manual_seed` đúng một lần lúc định nghĩa cell nên mỗi fold khởi tạo
từ một RNG state khác nhau; sửa xong, phương sai giữa các fold sụp từ **±0.0393 xuống ±0.0141**.

**Mỗi fold train đúng `BEST_EPOCH`** — epoch mà mục 6 đã chọn bằng val — **không chạy hết
`cfg.epochs`.** `cfg.loss_floor` (hằng số trước đây chọn bằng cách nhìn gold) đã **bỏ hẳn**
ở [`TRAIN.md`](../docs/TRAIN.md) muc 14.2. Các fold vẫn hội tụ nhanh chậm khác nhau ở cùng
số epoch đó — phương sai còn lại phản ánh đúng dữ liệu nhỏ, chứ không phải nhiễu do bị cắt ở
những mức fit khác hẳn nhau như bản `loss_floor` cũ.

Lịch learning-rate vẫn dựng dài `cfg.epochs` (`sched_epochs`), giống hệt mục 6, để quỹ đạo lr
của CV và của mục 6 là một.

**Đây là con số để báo cáo trên silver.** Cell dưới dùng `cfg_xlmr` (tham chiếu); mục 9.b có
CV riêng cho mDeBERTa (backbone chính) ngay sau nó để so trực tiếp. Con số CUỐI CÙNG ở mục 10.


In [20]:
N_FOLDS = FOLDS["n_folds"]
fold_of = FOLDS["fold_of_pair_id"]
# CV không cắt theo số epoch DÀI cfg_xlmr.epochs (20) mà theo BEST_EPOCH mà mục 6 đã chọn
# bằng val -- TRAIN.md muc 14.2: cfg.loss_floor (hằng số chọn bằng cách nhìn gold) đã bỏ
# hẳn, thay bằng "mỗi fold train đúng BEST_EPOCH, trần epoch duy nhất". Trước bản này biến
# này không được nối vào đâu cả nên mỗi fold âm thầm chạy hết 20 epoch không dừng sớm.
# Nếu mục 6 chưa chạy thì BEST_EPOCH chưa tồn tại -- tạm lấy 3 để cell này không crash.
# sched_epochs=cfg_xlmr.epochs giữ nguyên quỹ đạo learning-rate DÀI mà mục 6 đã dùng khi
# chọn BEST_EPOCH -- nếu không thì lr trong CV decay về 0 sớm hơn hẳn, hai số hết so được.
# cv_cfg dùng cfg_xlmr (không phải cfg trần) vì mục 8 là CV THAM CHIẾU cho xlm-r -- cfg giờ
# mặc định là mDeBERTa (muc 15), và REPORT["model:..."] ở mục 7/10 đã có nghĩa cố định là
# "model xlm-r của mục 6". CV cho mDeBERTa chạy ở cell riêng ngay sau mục 9.b.
_best_epoch_ref = globals().get("BEST_EPOCH", 3)
cv_cfg = replace(cfg_xlmr, epochs=_best_epoch_ref)
print(f"CV (xlm-r, mục 8 -- chỉ tham chiếu): mỗi fold train đúng {_best_epoch_ref} epoch "
      f"(BEST_EPOCH mục 6) | lịch lr dài {cfg_xlmr.epochs} epoch | "
      f"ghim vào train mọi vòng: {sum(1 for r in rows if fold_of[r['pair_id']] == -1)} cặp\n")

all_t, all_p, fold_f1 = [], [], []
for k in range(N_FOLDS):
    tr = [r for r in rows if fold_of[r["pair_id"]] != k]     # -1 rơi vào đây ở MỌI vòng
    te = [r for r in rows if fold_of[r["pair_id"]] == k]
    nc = collections.Counter(r["relation"] for r in te)
    print(f"\n--- fold {k}: train={len(tr)} test={len(te)}  "
          + " ".join(f"{l[:4]}:{nc.get(l,0)}" for l in LABELS) + " ---")
    m_, t_, _ = train_model(tr, cv_cfg, tag=f"[f{k}] ", sched_epochs=cfg_xlmr.epochs)
    yp_ = predict_logits(m_, t_, te, cv_cfg.symmetric_tta).argmax(1)
    yt_ = np.array([L2I[r["relation"]] for r in te])
    s = f1_score(yt_, yp_, average="macro", zero_division=0); fold_f1.append(s)
    print(f"  fold {k} macro-F1 = {s:.4f}")
    all_t.append(yt_); all_p.append(yp_)
    del m_; torch.cuda.empty_cache()

cv_t, cv_p = np.concatenate(all_t), np.concatenate(all_p)
print("\n" + "="*62)
print(f"macro-F1 từng fold: {[f'{s:.3f}' for s in fold_f1]}")
print(f"trung bình = {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
print("="*62)
print(classification_report(cv_t, cv_p, target_names=LABELS, digits=3, zero_division=0))
dcv = np.array([axis_dist(LABELS[t],LABELS[p]) for t,p in zip(cv_t,cv_p)])
print(f"đúng hoặc lệch 1 bước: {(dcv<=1).mean():.3f}   sai nặng (>=2): {(dcv>=2).mean():.3f}")
q1_q2_split(cv_t, cv_p, tag="theo cây RUBRIC_GOLD:  ")

# Mốc so sánh, tính trên ĐÚNG tập vừa chấm -> so được trực tiếp với con số trên. Lớp đông
# nhất của TRAIN (MAJ_LABEL, mục 4) chứ không ghim cứng COMPLEMENTARY -- sau gán lại ở
# muc 10, lớp đông nhất là UNRELATED. Cùng lỗi đã sửa ở mục 4 (xem cell baseline).
maj = np.full(len(cv_t), L2I[MAJ_LABEL])
print(f"\nmajority baseline trên cùng tập: macro-F1 = "
      f"{f1_score(cv_t, maj, average='macro', zero_division=0):.4f}")
print("\n[!] Nếu ở trên có dòng 'CHƯA HỘI TỤ' thì độ lệch giữa các fold là nhiễu khởi tạo,")
print("    không phải phương sai dữ liệu — xem TRAIN.md muc 11.1 trước khi diễn giải.")

eval_views(cv_t, cv_p, store="model:CV-silver")


CV (xlm-r, mục 8 -- chỉ tham chiếu): mỗi fold train đúng 9 epoch (BEST_EPOCH mục 6) | lịch lr dài 20 epoch | ghim vào train mọi vòng: 39 cặp


--- fold 0: train=796 test=193  AGRE:14 PART:43 COMP:32 PART:34 CONT:4 UNRE:66 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [f0] epoch 1/9  loss=1.8117
  [f0] epoch 2/9  loss=1.7968
  [f0] epoch 3/9  loss=1.7672
  [f0] epoch 4/9  loss=1.6130
  [f0] epoch 5/9  loss=1.1773
  [f0] epoch 6/9  loss=0.8146
  [f0] epoch 7/9  loss=0.4745
  [f0] epoch 8/9  loss=0.2528
  [f0] epoch 9/9  loss=0.1274
  fold 0 macro-F1 = 0.2999

--- fold 1: train=803 test=186  AGRE:12 PART:39 COMP:28 PART:32 CONT:5 UNRE:70 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

  [f3] epoch 1/9  loss=1.8058
  [f3] epoch 2/9  loss=1.7837
  [f3] epoch 3/9  loss=1.5929
  [f3] epoch 4/9  loss=1.1975
  [f3] epoch 5/9  loss=0.7895
  [f3] epoch 6/9  loss=0.4488
  [f3] epoch 7/9  loss=0.2413
  [f3] epoch 8/9  loss=0.1233
  [f3] epoch 9/9  loss=0.0607
  fold 3 macro-F1 = 0.2468

--- fold 4: train=797 test=192  AGRE:15 PART:43 COMP:29 PART:32 CONT:4 UNRE:69 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [f4] epoch 1/9  loss=1.8105
  [f4] epoch 2/9  loss=1.7917
  [f4] epoch 3/9  loss=1.7193
  [f4] epoch 4/9  loss=1.5077
  [f4] epoch 5/9  loss=1.1595
  [f4] epoch 6/9  loss=0.8104
  [f4] epoch 7/9  loss=0.5150
  [f4] epoch 8/9  loss=0.3000
  [f4] epoch 9/9  loss=0.1676
  fold 4 macro-F1 = 0.2554

macro-F1 từng fold: ['0.300', '0.278', '0.261', '0.247', '0.255']
trung bình = 0.2682 ± 0.0189
                       precision    recall  f1-score   support

            AGREEMENT      0.278     0.143     0.189        70
    PARTIAL_AGREEMENT      0.351     0.476     0.404       210
        COMPLEMENTARY      0.223     0.208     0.215       149
PARTIAL_CONTRADICTION      0.325     0.325     0.325       160
        CONTRADICTION      0.000     0.000     0.000        22
            UNRELATED      0.522     0.499     0.510       339

             accuracy                          0.381       950
            macro avg      0.283     0.275     0.274       950
         weighted avg      0.374     0

{'full6': {'macro_f1': 0.2738002247783047,
  'acc': 0.38105263157894737,
  'majority_f1': 0.08766485647788984},
 'merge5': {'macro_f1': 0.3359740628374139,
  'acc': 0.39263157894736844,
  'majority_f1': 0.1051978277734678},
 'deploy3': {'macro_f1': 0.4855999237417838,
  'acc': 0.5178947368421053,
  'majority_f1': 0.20739666424945613}}

## 9 · Learning curve — *thêm dữ liệu có đáng không?*

Trả lời bằng số cho câu "1099 cặp đã đủ chưa": train trên 25/50/75/100% rồi nhìn độ dốc.
Còn dốc → gán thêm nhãn sẽ có lời. Đã phẳng → tiền nên đổ vào chỗ khác (model lớn hơn,
sửa rubric, cân bằng lớp).

In [21]:
curve = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    sub = subsample_groups(train_rows, frac) if frac < 1 else train_rows
    m_, t_, _ = train_model(sub, cfg, tag=f"[{int(frac*100)}%] ", val_rows=val_rows)
    yp_ = predict_logits(m_, t_, test_rows, cfg.symmetric_tta).argmax(1)
    s = f1_score(y_test, yp_, average="macro", zero_division=0)
    curve.append((len(sub), s)); print(f"  n={len(sub):<5} macro-F1={s:.4f}")
    del m_; torch.cuda.empty_cache()

print("\n n_train   macro-F1   Δ so với mức trước")
prev = None
for n_,s_ in curve:
    print(f"{n_:>8}{s_:>11.4f}" + (f"{s_-prev:>+12.4f}" if prev is not None else ""))
    prev = s_
print("\nΔ cuối còn lớn -> gán thêm nhãn còn lời. Δ ~0 -> đã bão hoà.")
print("⚠ CHƯA DÙNG ĐƯỜNG NÀY ĐỂ RA QUYẾT ĐỊNH. Hai lần chạy gần nhất:")
print("    2026-08-29a  0.226 / 0.169 / 0.220 / 0.268   (mức 50% thấp hơn mức 25%)")
print("    2026-08-29b  0.115 / 0.225 / 0.206 / 0.219   (mức 75% thấp hơn mức 50%)")
print("  Nhiễu giữa các lần chạy lớn hơn hiệu ứng của dữ liệu. Hai lý do, phải gỡ cả hai:")
print("    1. mỗi mức chỉ 1 seed, chấm trên test ~100 cặp -> cần >=3 seed, chấm bằng CV")
print("    2. lần 2026-08-29b mức 25% kết thúc ở loss 1.67, sát mức đoán bừa 1.79 — đường")
print("       cong khi đó đo optimizer thất bại chứ không đo giá trị của dữ liệu")
print("  Xem docs/TRAIN.md muc 9.5 và 11.7.")


[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([6])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [25%] chọn epoch theo macro-F1 VÀ val loss trên 5/6 lớp — bỏ CONTRADICTION(n=2) | biên 0.01 | patience 4
  [25%] epoch 1/20  loss=1.8203  val6=0.1027  val_sel=0.1232  val_loss=1.7915  *
  [25%] epoch 2/20  loss=1.7720  val6=0.2759  val_sel=0.2311  val_loss=1.7220  *
  [25%] epoch 3/20  loss=1.6416  val6=0.2004  val_sel=0.2405  val_loss=1.6878
  [25%] epoch 4/20  loss=1.4111  val6=0.2395  val_sel=0.2875  val_loss=1.6012  *
  [25%] epoch 5/20  loss=1.1392  val6=0.3583  val_sel=0.4300  val_loss=1.5682  *
  [25%] epoch 6/20  loss=0.8944  val6=0.3368  val_sel=0.4042  val_loss=1.5790
  [25%] epoch 7/20  loss=0.7355  val6=0.3681  val_sel=0.4418  val_loss=1.6562  *
  [25%] epoch 8/20  loss=0.6005  val6=0.3403  val_sel=0.3750  val_loss=1.6369
  [25%] epoch 9/20  loss=0.4612  val6=0.3325  val_sel=0.3990  val_loss=1.6424
  [25%] dừng sớm: val loss không giảm sau 4 epoch liền
  [25%] -> giữ epoch 7 (val_sel 0.4418, train loss tại đó 0.7355)
  n=197   macro-F1=0.3066


[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


...

  [75%] chọn epoch theo macro-F1 VÀ val loss trên 5/6 lớp — bỏ CONTRADICTION(n=2) | biên 0.01 | patience 4
  [75%] epoch 1/20  loss=1.8134  val6=0.1620  val_sel=0.1944  val_loss=1.7559  *
  [75%] epoch 2/20  loss=1.6792  val6=0.2951  val_sel=0.3541  val_loss=1.4906  *
  [75%] epoch 3/20  loss=1.2750  val6=0.3147  val_sel=0.3333  val_loss=1.3133
  [75%] epoch 4/20  loss=0.8787  val6=0.4081  val_sel=0.4097  val_loss=1.2839  *
  [75%] epoch 5/20  loss=0.5748  val6=0.3379  val_sel=0.3655  val_loss=1.4477
  [75%] epoch 6/20  loss=0.3598  val6=0.3860  val_sel=0.4060  val_loss=1.6388
  [75%] epoch 7/20  loss=0.2000  val6=0.4127  val_sel=0.4225  val_loss=1.7137  *
  [75%] epoch 8/20  loss=0.0956  val6=0.3632  val_sel=0.3914  val_loss=2.0016
  [75%] dừng sớm: val loss không giảm sau 4 epoch liền
  [75%] -> giữ epoch 7 (val_sel 0.4225, train loss tại đó 0.2000)
  n=606   macro-F1=0.3582


[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([6])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [100%] chọn epoch theo macro-F1 VÀ val loss trên 5/6 lớp — bỏ CONTRADICTION(n=2) | biên 0.01 | patience 4
  [100%] epoch 1/20  loss=1.8179  val6=0.2213  val_sel=0.2656  val_loss=1.7395  *
  [100%] epoch 2/20  loss=1.5918  val6=0.2541  val_sel=0.3049  val_loss=1.3341  *
  [100%] epoch 3/20  loss=1.1228  val6=0.4059  val_sel=0.4337  val_loss=1.1510  *
  [100%] epoch 4/20  loss=0.7031  val6=0.3426  val_sel=0.4112  val_loss=1.3194
  [100%] epoch 5/20  loss=0.4655  val6=0.5145  val_sel=0.5374  val_loss=1.3093  *
  [100%] epoch 6/20  loss=0.3152  val6=0.4543  val_sel=0.4651  val_loss=1.5986
  [100%] epoch 7/20  loss=0.1688  val6=0.5368  val_sel=0.5642  val_loss=1.8199  *
  [100%] dừng sớm: val loss không giảm sau 4 epoch liền
  [100%] -> giữ epoch 7 (val_sel 0.5642, train loss tại đó 0.1688)
  n=788   macro-F1=0.3777

 n_train   macro-F1   Δ so với mức trước
     197     0.3066
     396     0.4296     +0.1231
     606     0.3582     -0.0714
     788     0.3777     +0.0195

Δ cuối còn lớn -

## 9.b · Fine-tune mDeBERTa-v3-base-xnli — ablation NLI warm-up

**Câu hỏi:** mDeBERTa zero-shot (mục 4.b) đang thắng xlm-r fine-tuned. Fine-tune
chính model đó trên silver data có thêm được gì không? So sánh kiểm soát đúng **một biến**:

| | zero-shot | fine-tuned trên silver |
|---|---|---|
| xlm-r-base | — | ✓ mục 6 |
| **mDeBERTa-v3-base-xnli** | ✓ mục 4.b | → **ô này** |

- Fine-tune **thắng** zero-shot → silver data có giá trị thực.
- Fine-tune **≈ hoặc thua** zero-shot → silver data đang làm hại; cần nhìn lại chất lượng nhãn.

5-fold CV với mDeBERTa chạy ở **cell kế tiếp** (dùng cùng folds và logic với muc 8) để có số so sánh trực tiếp xlm-r vs mDeBERTa trên silver.

In [22]:
# cfg đã dùng mDeBERTa làm backbone mặc định (xem muc 2, Cfg.model).
# cfg_deb giữ lại để các cell sau (CV, gold eval, bảng tổng hợp) không cần sửa tên biến.
cfg_deb = cfg

model_deb, tok_deb, BEST_EPOCH_DEB = train_model(train_rows, cfg_deb, val_rows=val_rows)
logits_test_deb = predict_logits(model_deb, tok_deb, test_rows, cfg_deb.symmetric_tta)
y_pred_deb = logits_test_deb.argmax(1)

print(f"\nmacro-F1 (test-EN, mDeBERTa fine-tuned) = "
      f"{f1_score(y_test, y_pred_deb, average='macro', zero_division=0):.4f}")
print(f"BEST_EPOCH_DEB = {BEST_EPOCH_DEB}")
eval_views(y_test, y_pred_deb, tag="[test-EN] ", store="model_deb:test-EN")

# --- Gold: toàn bộ 129 cặp ---
y_gold_deb = np.array([L2I[r["relation"]] for r in gold_rows])
logits_gold_deb = predict_logits(model_deb, tok_deb, gold_rows, cfg_deb.symmetric_tta)
p_gold_deb = logits_gold_deb.argmax(1)
eval_views(y_gold_deb, p_gold_deb, tag="[gold] ", store="model_deb:gold")

# --- Flip-rate: mDeBERTa có đối xứng hơn xlm-r không? ---
raw_fwd_deb = predict_logits(model_deb, tok_deb, test_rows, symmetric_tta=False)
swapped_deb = [{**r, "left": r["right"], "right": r["left"]} for r in test_rows]
raw_rev_deb = predict_logits(model_deb, tok_deb, swapped_deb, symmetric_tta=False)
flip_deb = (raw_fwd_deb.argmax(1) != raw_rev_deb.argmax(1))
REPORT["flip:test-EN-deb"] = float(flip_deb.mean())
print(f"\nflip-rate mDeBERTa fine-tuned (thô, không TTA): "
      f"{flip_deb.mean():.3f}  ({flip_deb.sum()}/{len(flip_deb)})")
print("đối chiếu xlm-r fine-tuned: xem mục 7.5 + bảng mục 11")

# --- Flip-rate trên GOLD: model_deb là backbone thật đưa vào workflow (muc 15), nên đây
# là con số đối xứng thật để báo cáo -- không chỉ test-EN. ---
raw_fwd_deb_g = predict_logits(model_deb, tok_deb, gold_rows, symmetric_tta=False)
swapped_deb_g = [{**r, "left": r["right"], "right": r["left"]} for r in gold_rows]
raw_rev_deb_g = predict_logits(model_deb, tok_deb, swapped_deb_g, symmetric_tta=False)
flip_deb_g = (raw_fwd_deb_g.argmax(1) != raw_rev_deb_g.argmax(1))
REPORT["flip:gold-deb"] = float(flip_deb_g.mean())
print(f"flip-rate mDeBERTa fine-tuned trên GOLD (thô, không TTA): "
      f"{flip_deb_g.mean():.3f}  ({flip_deb_g.sum()}/{len(flip_deb_g)})")

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([6])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  trọng số lớp (đích=uniform): AGREEM=1.21  PARTIA=0.43  COMPLE=0.60  PARTIA=0.55  CONTRA=2.94  UNRELA=0.27
  chọn epoch theo macro-F1 VÀ val loss trên 5/6 lớp — bỏ CONTRADICTION(n=2) | biên 0.01 | patience 4
  epoch 1/20  loss=1.8179  val6=0.2213  val_sel=0.2656  val_loss=1.7395  *
  epoch 2/20  loss=1.5918  val6=0.2541  val_sel=0.3049  val_loss=1.3341  *
  epoch 3/20  loss=1.1228  val6=0.4059  val_sel=0.4337  val_loss=1.1510  *
  epoch 4/20  loss=0.7031  val6=0.3426  val_sel=0.4112  val_loss=1.3194
  epoch 5/20  loss=0.4655  val6=0.5145  val_sel=0.5374  val_loss=1.3093  *
  epoch 6/20  loss=0.3152  val6=0.4543  val_sel=0.4651  val_loss=1.5986
  epoch 7/20  loss=0.1688  val6=0.5368  val_sel=0.5642  val_loss=1.8199  *
  dừng sớm: val loss không giảm sau 4 epoch liền
  -> giữ epoch 7 (val_sel 0.5642, train loss tại đó 0.1688)

macro-F1 (test-EN, mDeBERTa fine-tuned) = 0.3777
BEST_EPOCH_DEB = 7

[test-EN] mức nhãn      n lớp   macro-F1      acc  maj(oracle)
          full6             6 

In [23]:
# 5-fold CV cho mDeBERTa -- mirror hệt cell CV ở mục 8, khác backbone. TRAIN.md muc 15.1
# bước 3: "Mục 8 (CV xlm-r, chỉ tham chiếu) + cell 44 (CV mDeBERTa) -- đọc kết quả silver,
# KHÔNG mở mục 10." Cùng folds, cùng lr schedule dài cfg_deb.epochs, cùng cách cắt theo
# BEST_EPOCH (muc 14.2) để hai con số so được trực tiếp.
N_FOLDS = FOLDS["n_folds"]
fold_of = FOLDS["fold_of_pair_id"]
_best_epoch_deb_ref = globals().get("BEST_EPOCH_DEB", 3)
cv_cfg_deb = replace(cfg_deb, epochs=_best_epoch_deb_ref)
print(f"CV (mDeBERTa): mỗi fold train đúng {_best_epoch_deb_ref} epoch (BEST_EPOCH_DEB mục "
      f"9.b) | lịch lr dài {cfg_deb.epochs} epoch | "
      f"ghim vào train mọi vòng: {sum(1 for r in rows if fold_of[r['pair_id']] == -1)} cặp\n")

all_t_deb, all_p_deb, fold_f1_deb = [], [], []
for k in range(N_FOLDS):
    tr = [r for r in rows if fold_of[r["pair_id"]] != k]
    te = [r for r in rows if fold_of[r["pair_id"]] == k]
    nc = collections.Counter(r["relation"] for r in te)
    print(f"\n--- fold {k}: train={len(tr)} test={len(te)}  "
          + " ".join(f"{l[:4]}:{nc.get(l,0)}" for l in LABELS) + " ---")
    m_, t_, _ = train_model(tr, cv_cfg_deb, tag=f"[f{k}] ", sched_epochs=cfg_deb.epochs)
    yp_ = predict_logits(m_, t_, te, cv_cfg_deb.symmetric_tta).argmax(1)
    yt_ = np.array([L2I[r["relation"]] for r in te])
    s = f1_score(yt_, yp_, average="macro", zero_division=0); fold_f1_deb.append(s)
    print(f"  fold {k} macro-F1 = {s:.4f}")
    all_t_deb.append(yt_); all_p_deb.append(yp_)
    del m_; torch.cuda.empty_cache()

cv_t_deb, cv_p_deb = np.concatenate(all_t_deb), np.concatenate(all_p_deb)
print("\n" + "="*62)
print(f"macro-F1 từng fold: {[f'{s:.3f}' for s in fold_f1_deb]}")
print(f"trung bình = {np.mean(fold_f1_deb):.4f} ± {np.std(fold_f1_deb):.4f}")
print("="*62)
print(classification_report(cv_t_deb, cv_p_deb, target_names=LABELS, digits=3, zero_division=0))
dcv_deb = np.array([axis_dist(LABELS[t],LABELS[p]) for t,p in zip(cv_t_deb,cv_p_deb)])
print(f"đúng hoặc lệch 1 bước: {(dcv_deb<=1).mean():.3f}   sai nặng (>=2): {(dcv_deb>=2).mean():.3f}")
q1_q2_split(cv_t_deb, cv_p_deb, tag="theo cây RUBRIC_GOLD:  ")

maj_deb = np.full(len(cv_t_deb), L2I[MAJ_LABEL])
print(f"\nmajority baseline trên cùng tập: macro-F1 = "
      f"{f1_score(cv_t_deb, maj_deb, average='macro', zero_division=0):.4f}")
print("\n[!] Nếu ở trên có dòng 'CHƯA HỘI TỤ' thì độ lệch giữa các fold là nhiễu khởi tạo,")
print("    không phải phương sai dữ liệu — xem TRAIN.md muc 11.1 trước khi diễn giải.")

eval_views(cv_t_deb, cv_p_deb, store="model_deb:CV-silver")


CV (mDeBERTa): mỗi fold train đúng 7 epoch (BEST_EPOCH_DEB mục 9.b) | lịch lr dài 20 epoch | ghim vào train mọi vòng: 39 cặp


--- fold 0: train=796 test=193  AGRE:14 PART:43 COMP:32 PART:34 CONT:4 UNRE:66 ---


[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([6])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [f0] epoch 1/7  loss=1.8049
  [f0] epoch 2/7  loss=1.6014
  [f0] epoch 3/7  loss=1.1162
  [f0] epoch 4/7  loss=0.7355
  [f0] epoch 5/7  loss=0.4932
  [f0] epoch 6/7  loss=0.3716
  [f0] epoch 7/7  loss=0.2925
  fold 0 macro-F1 = 0.4781

--- fold 1: train=803 test=186  AGRE:12 PART:39 COMP:28 PART:32 CONT:5 UNRE:70 ---


...

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([6])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [f4] epoch 1/7  loss=1.8112
  [f4] epoch 2/7  loss=1.5954
  [f4] epoch 3/7  loss=1.1244
  [f4] epoch 4/7  loss=0.7283
  [f4] epoch 5/7  loss=0.5021
  [f4] epoch 6/7  loss=0.3635
  [f4] epoch 7/7  loss=0.2793
  fold 4 macro-F1 = 0.3584

macro-F1 từng fold: ['0.478', '0.396', '0.430', '0.457', '0.358']
trung bình = 0.4239 ± 0.0427
                       precision    recall  f1-score   support

            AGREEMENT      0.472     0.357     0.407        70
    PARTIAL_AGREEMENT      0.447     0.586     0.507       210
        COMPLEMENTARY      0.304     0.282     0.293       149
PARTIAL_CONTRADICTION      0.557     0.550     0.553       160
        CONTRADICTION      0.304     0.318     0.311        22
            UNRELATED      0.545     0.487     0.514       339

             accuracy                          0.474       950
            macro avg      0.438     0.430     0.431       950
         weighted avg      0.477     0.474     0.472       950

đúng hoặc lệch 1 bước: 0.777   sai

{'full6': {'macro_f1': 0.4308320681520051,
  'acc': 0.47368421052631576,
  'majority_f1': 0.08766485647788984},
 'merge5': {'macro_f1': 0.475213912244835,
  'acc': 0.49894736842105264,
  'majority_f1': 0.1051978277734678},
 'deploy3': {'macro_f1': 0.6208011682254977,
  'acc': 0.6252631578947369,
  'majority_f1': 0.20739666424945613}}

## 10 · Đánh giá trên GOLD — con số duy nhất có nền người

`golden_set/gold_test.jsonl`: 129 cặp, `HUMAN_VERIFIED`, annotator `NTH`, phân bố **cân đều
cả 6 lớp** (COMP 29 · PART_CONTRA 24 · PART_AGREE 24 · AGREE 20 · UNREL 18 · CONTRA 14).

Mọi con số ở các mục trên đo **độ khớp với nhãn Claude**. Đây là chỗ duy nhất đo **độ khớp
với người**. Nếu hai con số lệch xa nhau thì cái đáng tin là con số ở đây.

Ba khoảng cách phải đọc kèm, nếu không sẽ quy sai nguyên nhân:

| | train (silver) | eval (gold) |
|---|---|---|
| Ngôn ngữ | 989/989 tiếng Anh | **129/129 tiếng Việt** |
| Miền | review paper ICLR về ML | phản biện đề tài đại học VN (3 cohort, 17 tiêu chí) |
| Nhãn | rubric v1 | contract gold ([`RUBRIC_GOLD.md`](../docs/RUBRIC_GOLD.md)) |

Nên đây là **cross-lingual zero-shot + đổi miền**. Kết quả 2026-08-29b: gold 0.2548 vs test
tiếng Anh 0.2932 — chênh chỉ 0.038, tức `xlm-roberta-base` đã đóng được khoảng cách thứ
nhất. Khoảng cách thứ ba xử lý bằng việc gán lại silver. Khoảng cách thứ hai thì chưa, và
bảng theo cohort ở dưới cho biết nó tốn bao nhiêu (acc 0.216 / 0.385 / 0.308 — chênh gần 2×).

**Còn một khoảng cách thứ tư mà bảng trên không nêu: lệch tiên nghiệm.** Silver có 34.6%
UNRELATED, gold chỉ 14.0%, và model đoán UNRELATED 35.7% trên gold — nó tái tạo phân bố đã
học. Cell dưới báo cả ba con số: **thô** (số báo cáo), **bù prior đều** (label-free, cũng
báo cáo được), và **bù prior gold** (oracle, chỉ để biết trần).


In [24]:
# gold_rows: toan bo 129 cap, nap o muc 3.b, khong chia cohort.
y_gold = np.array([L2I[r["relation"]] for r in gold_rows])
logits_gold = predict_logits(model, tok, gold_rows, cfg.symmetric_tta)
p_gold = logits_gold.argmax(1)
K = len(LABELS)
def _mf1(y, p): return f1_score(y, p, average="macro", zero_division=0)

print("="*74)
print(f"GOLD -- tieng Viet, HUMAN_VERIFIED (annotator NTH), n={len(gold_rows)}")
print("="*74)
eval_views(y_gold, p_gold, tag="[gold] ", store="model:gold")

print(classification_report(y_gold, p_gold, labels=list(range(K)),
                            target_names=LABELS, digits=3, zero_division=0))
maj_g = np.full(len(y_gold), collections.Counter(y_gold).most_common(1)[0][0])
print(f"macro-F1 (gold, tho) = {_mf1(y_gold, p_gold):.4f}")
print(f"majority baseline tren gold = {_mf1(y_gold, maj_g):.4f}")

# ---------------------------------------------------------------------------
# 10.b . Bu lech tien nghiem
# ---------------------------------------------------------------------------
gold_prior = collections.Counter(r["relation"] for r in gold_rows)
p_unif = prior_shift_logits(logits_gold, train_rows, None,       cfg.prior_tau).argmax(1)
p_orac = prior_shift_logits(logits_gold, train_rows, gold_prior, cfg.prior_tau).argmax(1)

print("\n" + ("bu prior (tau=%.1f)" % cfg.prior_tau).ljust(34) + "macro-F1".rjust(10) + "acc".rjust(9))
for nm, pp in [("tho, khong bu", p_gold),
               ("prior DEU (label-free, bao cao duoc)", p_unif),
               ("prior GOLD (oracle, chi la tran)", p_orac)]:
    print(f"{nm:<34}{_mf1(y_gold, pp):>10.4f}{(y_gold==pp).mean():>9.4f}")

print("\n" + "lop".ljust(24) + "prior train".rjust(12) + "that (gold)".rjust(13)
      + "model doan".rjust(12) + "ti le".rjust(8))
_ptr = collections.Counter(r["relation"] for r in train_rows)
for i, l in enumerate(LABELS):
    t_ = (y_gold == i).sum(); q_ = (p_gold == i).sum()
    tl = _ptr.get(l,0)/len(train_rows); gl = t_/len(y_gold); ml = q_/len(y_gold)
    ratio = q_/t_ if t_ else float("nan")
    print(f"{l:<24}{tl:>11.1%}{gl:>13.1%}{ml:>12.1%}{ratio:>7.2f}x")

dg = np.array([axis_dist(LABELS[t], LABELS[p]) for t,p in zip(y_gold, p_gold)])
print(f"\ndung tuyet doi {(dg==0).mean():.3f}  |  dung-hoac-lech-1-buoc {(dg<=1).mean():.3f}"
      f"  |  sai nang (>=2) {(dg>=2).mean():.3f}")
q1_q2_split(y_gold, p_gold, tag="theo cay RUBRIC_GOLD:  ")

cmg = confusion_matrix(y_gold, p_gold, labels=list(range(K)))
cma = confusion_matrix(y_gold, p_unif, labels=list(range(K)))
for title, cm_ in [("THO", cmg), ("SAU KHI BU PRIOR DEU", cma)]:
    print(f"\nma tran nham lan -- {title}")
    print(" "*26 + "".join(f"{l[:6]:>8}" for l in LABELS) + "     n")
    for i,l in enumerate(LABELS):
        print(f"{l:<26}" + "".join(f"{v:>8}" for v in cm_[i]) + f"  {cm_[i].sum():>5}")

def _top_cells(cm_, k=6):
    return sorted(((cm_[i][j], LABELS[i], LABELS[j]) for i in range(K) for j in range(K)
                   if i != j and cm_[i][j] > 0), reverse=True)[:k]
print("\nO nham nhieu nhat (sau khi bu prior deu):")
for n_,t_,p_ in _top_cells(cma):
    q_branch = "sai Q1" if branch_of(t_) != branch_of(p_) else "sai Q2"
    print(f"  {n_:>3}x  {t_} -> {p_}   (cach {axis_dist(t_,p_)} buoc, {q_branch})")

i_c, i_u = L2I["COMPLEMENTARY"], L2I["UNRELATED"]
cu_adj  = cma[i_c][i_u] + cma[i_u][i_c]
top_adj = _top_cells(cma, 1)[0][0] if _top_cells(cma, 1) else 0
print()
if cu_adj >= top_adj and top_adj > 0:
    print("  Warning COMPLEMENTARY <-> UNRELATED VAN dan dau SAU khi bu prior")
else:
    print(f"  [OK] bu prior xong COMP<->UNREL khong dan dau ({cu_adj} vs {top_adj})")

print("\n" + "cohort".ljust(34) + "n".rjust(4) + "acc".rjust(8) + "macroF1".rjust(9))
by_c = collections.defaultdict(list)
for i,r in enumerate(gold_rows): by_c[r["paper_id"]].append(i)
for c, idx in sorted(by_c.items()):
    yt_, yp_ = y_gold[idx], p_gold[idx]
    print(f"{c:<34}{len(idx):>4}{(yt_==yp_).mean():>8.3f}{_mf1(yt_,yp_):>9.3f}")

# flip-rate tren gold
raw_f = predict_logits(model, tok, gold_rows, symmetric_tta=False)
swap_g = [{**r, "left": r["right"], "right": r["left"]} for r in gold_rows]
raw_r = predict_logits(model, tok, swap_g, symmetric_tta=False)
fl = (raw_f.argmax(1) != raw_r.argmax(1))
print(f"\nflip-rate tren gold (tho, khong TTA): {fl.mean():.3f}  ({fl.sum()}/{len(fl)})")
REPORT["flip:gold"] = float(fl.mean())


GOLD -- tieng Viet, HUMAN_VERIFIED (annotator NTH), n=129

[gold] mức nhãn      n lớp   macro-F1      acc  maj(oracle)
       full6             6     0.1886   0.2558       0.0612
       merge5            5     0.2333   0.2713       0.0910
       deploy3           3     0.4123   0.4806       0.2409
                       precision    recall  f1-score   support

            AGREEMENT      0.500     0.150     0.231        20
    PARTIAL_AGREEMENT      0.220     0.542     0.313        24
        COMPLEMENTARY      0.000     0.000     0.000        29
PARTIAL_CONTRADICTION      0.188     0.125     0.150        24
        CONTRADICTION      0.000     0.000     0.000        14
            UNRELATED      0.304     0.778     0.438        18

             accuracy                          0.256       129
            macro avg      0.202     0.266     0.189       129
         weighted avg      0.196     0.256     0.183       129

macro-F1 (gold, tho) = 0.1886
majority baseline tren gold = 0.0612



## 11 · Bảng tổng hợp — *số để đưa vào báo cáo*

Gom mọi thứ đã chấm ở trên thành một bảng. **In cả ba mức nhãn**: `full6` là con số
trung thực nhất và thấp nhất, `deploy3` là con số trả lời "dùng được chưa". Bỏ `full6`
đi thì câu hỏi đầu tiên nhận được sẽ là *"em chọn 3 lớp sau khi nhìn kết quả à?"*.

In [25]:
# ---------------------------------------------------------------------------
# Bảng 1 — MODEL vs BASELINE, ở cả ba mức nhãn
# ---------------------------------------------------------------------------
_ROWS = [("majority (lớp đông nhất)", "baseline:majority:{t}"),
         ("stance-rule (luật 3 dòng)", "baseline:stance-rule:{t}"),
         ("NLI zero-shot (mDeBERTa-xnli)", "nli:{t}"),
         ("GPT-4o-mini zero-shot", "gpt4om:{t}"),
         ("fine-tune xlm-r (của ta)", "model:{t}"),
         ("fine-tune mDeBERTa-xnli (của ta)", "model_deb:{t}")]
_TABS = [("test-EN", "test-EN"), ("gold", "gold")]
# NLI zero-shot chỉ có ở cột deploy3 và đó KHÔNG phải thiếu sót của phép so: model NLI có
# đúng 3 nhãn nên nó không diễn đạt nổi contract 6 lớp (không có khái niệm "cùng chiều
# nhưng khác phạm vi"). Ô "—" ở full6/merge5 chính là một kết quả cần báo cáo.

def _get(key, view):
    d = REPORT.get(key)
    return d.get(view, {}).get("macro_f1") if d else None

for view in ["merge5"]:  # chỉ báo cáo merge5; full6 & deploy3 bỏ để gọn
    print("\n" + "="*78)
    print(f"macro-F1 @ {view}  ({len(VIEW_LABELS[view])} lớp: {', '.join(VIEW_LABELS[view])})")
    print("="*78)
    print(f"{'':<32}" + "".join(f"{t:>15}" for t, _ in _TABS))
    for name, kt in _ROWS:
        cells = []
        for _, tag in _TABS:
            v = _get(kt.format(t=tag), view)
            cells.append(f"{v:>15.4f}" if v is not None else f"{'—':>15}")
        print(f"{name:<32}" + "".join(cells))
    # Δ cụ thể — 3 phép so có giá trị trong báo cáo:
    #   (1) xlm-r fine-tune vs mDeBERTa zero-shot      → fine-tune có hơn NLI sẵn không?
    #   (2) mDeBERTa fine-tune vs mDeBERTa zero-shot   → fine-tune backbone đó được bao nhiêu?
    #   (3) mDeBERTa fine-tune vs GPT-4o-mini zero-shot → tiệm cận LLM được không?
    print()
    _cmp_pairs = [
        ("fine-tune xlm-r",         "model:{t}",     "mDeBERTa zero-shot", "nli:{t}"),
        ("fine-tune mDeBERTa-xnli", "model_deb:{t}", "mDeBERTa zero-shot", "nli:{t}"),
        ("fine-tune mDeBERTa-xnli", "model_deb:{t}", "GPT-4o-mini zero-shot", "gpt4om:{t}"),
    ]
    for our_name, our_ktpl, ref_name, ref_ktpl in _cmp_pairs:
        for _, tag in _TABS:
            ours = _get(our_ktpl.format(t=tag), view)
            ref  = _get(ref_ktpl.format(t=tag), view)
            if ours is None or ref is None: continue
            ratio = f"  ({ours/ref:.2f}x)" if ref > 0 else ""
            print(f"  {our_name} vs {ref_name} @ {tag:<13}  "
                  f"{ours:.4f} vs {ref:.4f}  -> {ours-ref:+.4f}{ratio}")

# ---------------------------------------------------------------------------
# Bảng 2 — ĐỐI XỨNG: thứ đã giết pipeline Track B gốc
# ---------------------------------------------------------------------------
# TRAIN.md muc 0: 94% số cặp phải đi debate KHÔNG phải vì ba model bất đồng, mà vì MỘT
# model tự mâu thuẫn khi đảo thứ tự hai claim. Đó là lỗi đã làm hỏng cả Track B, và mốc
# <10% đặt ở muc 4.5 là định nghĩa của "đã khắc phục".
print("\n" + "="*78)
print("flip-rate (đảo thứ tự A/B, logit THÔ không TTA) — thấp hơn = đối xứng hơn")
print("="*78)
for nm, v in [("Qwen3-14B  (ensemble gốc)", 0.50), ("Gemma-2-9B (ensemble gốc)", 0.44),
              ("SeaLLM-v3-7B (ensemble gốc)", 0.18),
              ("GPT-4o-mini zero-shot — test EN", REPORT.get("flip:test-EN-gpt")),
              ("GPT-4o-mini zero-shot — gold VI", REPORT.get("flip:gold-gpt")),
              ("fine-tune xlm-r — test EN", REPORT.get("flip:test-EN")),
              ("fine-tune xlm-r — gold VI", REPORT.get("flip:gold")),
              ("fine-tune mDeBERTa-xnli — test EN", REPORT.get("flip:test-EN-deb")),
              ("fine-tune mDeBERTa-xnli — gold VI", REPORT.get("flip:gold-deb"))]:
    mark = ""
    if v is not None and ("fine-tune" in nm or "GPT" in nm):
        mark = "  <- ĐẠT MỐC <10%" if v < 0.10 else "  (mốc: <10%)"
    print(f"{nm:<34}{v:>8.3f}{mark}" if v is not None else f"{nm:<34}{'—':>8}")

# ---------------------------------------------------------------------------
# Bảng 3 — những gì PHẢI khai kèm, nếu không thì bảng trên đọc sai
# ---------------------------------------------------------------------------
print("\n" + "="*78)
print("KHAI KÈM")
print("="*78)
print(f"- LABEL_VIEW train = {LABEL_VIEW}; các mức thô hơn là ĐỌC LẠI cùng một dự đoán,")
print(f"  không phải train riêng. Model xuất {len(LABELS)} lớp.")
print(f"- gold n={len(gold_rows)} cặp, toàn bộ dùng để báo cáo (không chia hold-out).")
print(f"- trọng số lớp nhắm vào prior: {cfg.weight_target}")
print(f"- nhãn silver do LLM gán, chưa có annotator thứ hai -> CHƯA CÓ TRẦN κ. Không kết")
print(f"  luận được con số nào là 'gần trần' hay 'còn xa'. Đây là thiếu sót lớn nhất còn lại.")
if LABEL_VIEW == "full6":
    print(f"- ở full6, lớp CONTRADICTION chỉ 22 mẫu CV và F1 thường = 0.000; macro-F1 chia đều")
    print(f"  cho nó nên full6 là cận DƯỚI của năng lực model (TRAIN.md muc 11.6).")
for _cv_name, _cv_key in [("xlm-r", "model:CV-silver"), ("mDeBERTa", "model_deb:CV-silver")]:
    _cv = REPORT.get(_cv_key)
    if _cv:
        print(f"\n5-fold CV trên silver ({_cv_name}, mục 8/9.b) — số phát triển, KHÔNG phải số báo cáo:")
        for v, d in _cv.items():
            print(f"    {v:<10} macro-F1 {d['macro_f1']:.4f}   acc {d['acc']:.4f}")

print("\n(dán bảng này vào báo cáo kèm docs/TRAIN.md muc 0 cho phần 'pipeline gốc đã hỏng thế nào')")



macro-F1 @ merge5  (5 lớp: AGREEMENT, PARTIAL_AGREEMENT, COMPLEMENTARY, CONTRADICTION, UNRELATED)
                                        test-EN           gold
majority (lớp đông nhất)                 0.1072         0.0490
stance-rule (luật 3 dòng)                0.1983         0.2079
NLI zero-shot (mDeBERTa-xnli)            0.1823         0.1651
GPT-4o-mini zero-shot                    0.4055         0.5341
fine-tune xlm-r (của ta)                 0.4338         0.2333
fine-tune mDeBERTa-xnli (của ta)         0.3791         0.4191

  fine-tune xlm-r vs mDeBERTa zero-shot @ test-EN        0.4338 vs 0.1823  -> +0.2515  (2.38x)
  fine-tune xlm-r vs mDeBERTa zero-shot @ gold           0.2333 vs 0.1651  -> +0.0683  (1.41x)
  fine-tune mDeBERTa-xnli vs mDeBERTa zero-shot @ test-EN        0.3791 vs 0.1823  -> +0.1968  (2.08x)
  fine-tune mDeBERTa-xnli vs mDeBERTa zero-shot @ gold           0.4191 vs 0.1651  -> +0.2540  (2.54x)
  fine-tune mDeBERTa-xnli vs GPT-4o-mini zero-shot @ test-EN   

## 12 · Lưu checkpoint về Drive

Lưu thẳng **mDeBERTa** — backbone đã chốt cho workflow ở `TRAIN.md` mục 15 (so sánh
zero-shot + fine-tuned, không phải bằng cách nhìn gold). Bản trước có thêm một cell chọn
giữa xlm-r/mDeBERTa dựa trên macro-F1 trên gold rồi mới lưu — mâu thuẫn với chính nguyên
tắc "không để gold quyết định gì" ở mục 14.1, và đã bị bỏ (mục 16.2).


In [26]:
# Backbone đã chốt ở mục 15 (zero-shot + fine-tuned trên silver/gold, KHÔNG phải bằng
# cách chọn checkpoint theo gold) -- lưu thẳng mDeBERTa, không so sánh gì thêm ở đây.
from google.colab import drive
drive.mount('/content/drive')

OUT = "/content/drive/MyDrive/phase2_trackb/relation_classifier_v1"
model_deb.save_pretrained(OUT)
tok_deb.save_pretrained(OUT)

_cnt = collections.Counter(r["relation"] for r in train_rows)
train_prior = {l: round(_cnt.get(l, 0)/len(train_rows), 6) for l in LABELS}

with open(OUT + "/labels.json", "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS,
                "label_view": LABEL_VIEW,
                "view_map": {l: view_label(l) for l in FULL6},
                "backbone": cfg_deb.model,
                "weight_target": cfg_deb.weight_target,
                "dropped": [l for l in ALL_LABELS if l not in L2I],
                "symmetric_tta": cfg_deb.symmetric_tta,
                "tta_reduction": "mean",
                "best_epoch": BEST_EPOCH_DEB,
                "train_prior": train_prior,
                "split": "pipeline_data/processed/splits (stratified, few-shot ghim vào train)",
                "transformers": transformers.__version__}, f, ensure_ascii=False, indent=1)

print(f"đã lưu mDeBERTa (epoch {BEST_EPOCH_DEB}) -> {OUT}")
print(f"⚠ LỆCH TIÊN NGHIỆM: silver có {train_prior['UNRELATED']:.1%} UNRELATED; gold chỉ 14.0%.")
print("  Bù prior lúc suy luận: logit_adj = logit - tau*log(train_prior) + tau*log(prior_đích)")
print("  train_prior đã ghi sẵn trong labels.json.")


KeyboardInterrupt: 